In [203]:
# ------------------------------------------------- #
# 1- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 2- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 3- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 4- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 14 - Hibrido cosseno
# 15 - Hibrido pearson
# ------------------------------------------------- #

# 5- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 6- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 7- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 8- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 16 - Hibrido cosseno
# 17 - Hibrido pearson
# ------------------------------------------------- #

# 9- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *
# 10- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *
# 11 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *
# 12 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 18 - Hibrido cosseno
# 19 - Hibrido pearson
# ------------------------------------------------- #
# 13 - multiclasse
# ------------------------------------------------- #

# -> TODO: CORRIGIR/ATUALIZAR AS ANOTAÇÕES ACIMA


In [204]:
# -----------------------------
# bibliotecas -> TODO: Remover as que não uso
# -----------------------------
import random
import numpy as np
import json
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import os
import time
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, cohen_kappa_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# -----------------------------
# GRUPOS DE MÉTODOS
# -----------------------------
ITEM_PROFILE_METHODS = {1,2,5,6, 20, 21}
USER_PROFILE_METHODS = {3,4,7,8, 22, 23}
STRUCTURE_ONLY_METHODS = {9,10}
STRUCTURE_PERF_METHODS = {11,12,13}
HYBRID_PREFORM_METHODS = {14,15}
HYBRID_LEVEL_METHODS = {16,17}
HYBRID_STRUCTURE_METHODS = {18,19}


# -----------------------------
# PREPARANDO O DATASET
# -----------------------------

def initialize_data_frame():
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)
    sns.set(rc={'figure.figsize':(19.20, 10.80)})


def get_dungeon_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['dungeons']).T


def get_user_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['users']).T


def drop_invalid_rows(_data_frame):
    _data_frame.drop(_data_frame[_data_frame['UniqueRoomsEntered'] == 0].index, inplace=True)
    return _data_frame


def drop_unused_columns(_data_frame):
    _data_frame.drop('__collections__', inplace=True, axis=1)
    _data_frame = _data_frame.drop(list(_data_frame.filter(regex='^Total.*Quests')), axis=1)
    _data_frame.drop('CompletedQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedKillQuests', inplace=True, axis=1)
    _data_frame.rename(columns={'CompletedDamageQuests': 'CompletedKillQuests'}, inplace=True)
    _data_frame.drop('TreasuresCollected', inplace=True, axis=1)
    _data_frame.drop('TotalNpcs', inplace=True, axis=1)
    _data_frame.drop('WeaponName', inplace=True, axis=1)
    _data_frame.drop('HeatMap', inplace=True, axis=1)
    return _data_frame


def get_percentages(_data_frame):
    _data_frame['LockUsage'] = _data_frame['LocksOpened'].div(_data_frame['TotalLocks'].values)*100
    _data_frame['KeyCollection'] = _data_frame['KeysCollected'].div(_data_frame['TotalKeys'].values)*100
    _data_frame['MapCompletion'] = _data_frame['UniqueRoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['RoomRevisitRate'] = _data_frame['RoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['EnemyKillRate'] = _data_frame['EnemiesKilled'].div(_data_frame['TotalEnemies'].values)*100
    _data_frame['TotalAttempts'] = _data_frame['TotalWins'] + _data_frame['TotalDeaths'].values
    _data_frame['TotalAttempts'] = _data_frame['TotalAttempts'].astype(float)
    _data_frame['TotalWins'] = _data_frame['TotalWins'].astype(float)
    _data_frame['Success Rate'] = _data_frame['TotalAttempts'].div(_data_frame['TotalWins'].values)*100
    _data_frame.replace([np.inf, -np.inf], 0, inplace=True)
    return _data_frame


def drop_processed_columns(_data_frame):
    _data_frame.drop('LocksOpened', inplace=True, axis=1)
    _data_frame.drop('TotalLocks', inplace=True, axis=1)
    _data_frame.drop('KeysCollected', inplace=True, axis=1)
    _data_frame.drop('TotalKeys', inplace=True, axis=1)
    _data_frame.drop('EnemiesKilled', inplace=True, axis=1)
    _data_frame.drop('TotalEnemies', inplace=True, axis=1)
    _data_frame.drop('TotalAttempts', inplace=True, axis=1)
    _data_frame.drop('TotalWins', inplace=True, axis=1)
    _data_frame.drop('TotalDeaths', inplace=True, axis=1)
    _data_frame.drop('NpcsInteracted', inplace=True, axis=1)
    _data_frame.drop('RoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalRooms', inplace=True, axis=1)
    _data_frame.drop('UniqueRoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalTreasure', inplace=True, axis=1)
    return _data_frame


def split_preform_answers(_data_frame):
    _data_frame[['A', 'C', 'I', 'M']] = _data_frame['PreFormAnswers'].str.split(',', expand=True)
    _data_frame['A'] = _data_frame['A'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['C'] = _data_frame['C'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['I'] = _data_frame['I'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['M'] = _data_frame['M'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    return _data_frame

# -----------------------------
# DATA AUGMENTATION
# -----------------------------

def augment_dataframe_with_scaling(df, y, n, scale_factor):
    """
    Augments the DataFrame by adding `n` new rows, scaling numeric columns by a given factor.
    The new rows inherit the same 'y' values as their corresponding sampled rows.

    Parameters:
        df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        n (int): Number of new rows to generate.
        scale_factor (float): The factor by which numeric columns will be scaled.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify numeric columns, excluding 'y'
    numeric_columns = df.select_dtypes(include=['number']).columns.difference(y)
    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = df.sample(n=n, replace=True).reset_index(drop=True)

    # Scale numeric columns
    for col in numeric_columns:
        sampled_rows[col] *= scale_factor

    # Append the scaled rows to the original DataFrame
    augmented_df = pd.concat([df, sampled_rows], ignore_index=True)

    return augmented_df


def augment_data_random(_df, y, n):
    """
    Augments the DataFrame by adding `n` new rows.
    - Adds noise to numerical columns using a normal distribution.
    - Retains the same 'y' values for the newly added rows as their corresponding sampled rows.

    Parameters:
        _df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        y (list): the column(s) that will have their value retained
        n (int): Number of new rows to generate.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify float and integer columns, excluding 'y'
    float_columns = _df.select_dtypes(include=['float']).columns.difference(y)
    int_columns = _df.select_dtypes(include=['integer']).columns.difference(y)

    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = _df.sample(n=n, replace=True).reset_index(drop=True)

    # Add noise to float columns
    for col in float_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    for col in int_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    # Ensure integer columns remain integers
    sampled_rows[int_columns] = sampled_rows[int_columns].round().astype(int)

    # Append the sampled (augmented) rows to the original DataFrame
    augmented_df = pd.concat([_df, sampled_rows], ignore_index=True)
    return augmented_df

def leave_one_out_split(df, user_col='PlayerId'):
    train_list, test_list = [], []
    for u, grp in df.groupby(user_col):
        if len(grp)<2:
            train_list.append(grp); continue
        test_list.append(grp.sample(1, random_state=42))
        train_list.append(grp.drop(test_list[-1].index))
    return pd.concat(train_list), pd.concat(test_list)

In [205]:
# -----------------------------
# ESTRUTURA
# -----------------------------

def build_collab_df(_df, method, item_col_pass='PostFormAnswers'):
    # ----------------------------
    # DEFINIÇÃO DE COLUNAS E ITEM
    # ----------------------------
    if method in ITEM_PROFILE_METHODS:
      item_col = 'PreFormAnswers' if method in {1, 2} else 'LevelName'
      cols = ['PlayerId', item_col, 'PostFormAnswers']
      rating_type = "postform"

    elif method in USER_PROFILE_METHODS:
        item_col = 'PreFormAnswers' if method in {3, 4} else 'LevelName'
        cols = ['PlayerId', item_col,
                'Success Rate', 'TotalLostHealth',
                'EnemyKillRate', 'MapCompletion']
        rating_type = "performance"

    elif method in STRUCTURE_ONLY_METHODS.union(STRUCTURE_PERF_METHODS):
        cols = ['PlayerId', 'TotalRooms', 'TotalKeys']
        rating_type = "structure_only" if method in STRUCTURE_ONLY_METHODS else "structure_perf"

        if rating_type == "structure_only":
            cols.append('PostFormAnswers')
        else:
            cols += ['Success Rate', 'TotalLostHealth',
                     'EnemyKillRate', 'MapCompletion']

    elif method in HYBRID_PREFORM_METHODS.union(HYBRID_LEVEL_METHODS):
        # Define qual coluna será o Item
        item_col = 'PreFormAnswers' if method in HYBRID_PREFORM_METHODS else 'LevelName'

        cols = ['PlayerId', item_col, 'PostFormAnswers',
                'Success Rate', 'TotalLostHealth',
                'EnemyKillRate', 'MapCompletion']

        rating_type = "hybrid"

    elif method in HYBRID_STRUCTURE_METHODS:
        cols = ['PlayerId', 'TotalRooms', 'TotalKeys',
                'RoomsEntered', 'UniqueRoomsEntered',
                'PostFormAnswers', 'Success Rate',
                'TotalLostHealth', 'EnemyKillRate',
                'MapCompletion']
        rating_type = "hybrid_structure"

    else:
        raise ValueError(f"Método {method} inválido.")

    df = _df[cols].copy()
    df = df.rename(columns={'PlayerId': 'User'})

    if rating_type in ["structure_only", "structure_perf", "hybrid_structure"]:
        df['Item'] = build_item_from_structure(df)

    elif rating_type == "hybrid":
        df = df.rename(columns={item_col: 'Item'})

    else:
        df = df.rename(columns={item_col: 'Item'})

    RATING_STRATEGIES = {
        "postform": lambda df: df['PostFormAnswers'].apply(mean_post_forms),
        "structure_only": lambda df: df['PostFormAnswers'].apply(mean_post_forms),
        "performance": compute_performance_rating,
        "structure_perf": compute_performance_rating,
        "hybrid": compute_hybrid_rating,
        "hybrid_structure": compute_hybrid_rating
    }

    df['Rating'] = minmax(
        RATING_STRATEGIES[rating_type](df)
    )

    return df[['User', 'Item', 'Rating']]

def train_test_split_interactions(df, test_size=0.2):
    train, test = train_test_split(df, test_size=test_size, random_state=42)
    return train, test

In [206]:
# -----------------------------
# PREDICT
# -----------------------------
def weighted_mean(similarities, ratings):
    numerator = (similarities * ratings).sum()
    denominator = similarities.abs().sum()
    return numerator / denominator if denominator != 0 else 0




    return alpha * score_item + (1 - alpha) * score_user

def predict_cf(user, item, user_item_matrix, similarity_df, k=20, mode="user"):
          if mode == "user":
            if user not in similarity_df.index or item not in user_item_matrix.columns:
                return 0

            similar_entities = similarity_df.loc[user].drop(user, errors="ignore")
            ratings = user_item_matrix[item]

          elif mode == "item":
            if item not in similarity_df.columns or user not in user_item_matrix.index:
                return 0

            similar_entities = similarity_df[item]
            ratings = user_item_matrix.loc[user]

          else:
            raise ValueError("mode deve ser 'user' ou 'item'")
          # 👉 Filtrar itens não avaliados pelo usuário
          mask = ratings  != 0
          similar_entities = similar_entities[mask];
          ratings = ratings[mask]
          if similar_entities.empty:
              return 0
          # (Opcional) usar apenas os top-K vizinhos
          similar_entities  = similar_entities .sort_values(ascending=False).head(k)
          ratings = ratings.loc[similar_entities .index]

          return weighted_mean(similar_entities, ratings)

def predict_hybrid(user, item,
                   user_item_matrix,
                   item_similarity_df,
                   user_similarity_df,
                   alpha=0.5,k=20):

    score_item = predict_cf(
        user, item,
        user_item_matrix,
        item_similarity_df,
        mode="item",
        k=k
    )

    score_user = predict_cf(
        user, item,
        user_item_matrix,
        user_similarity_df,
        mode="user",
        k=k
    )

    return alpha * score_item + (1 - alpha) * score_user


def predict_all_users(user_item_matrix, similarity_matrix, method="user",alpha=0.5,k=20, item_similarity=None, user_similarity=None):
    predictions = []

    for user in user_item_matrix.index:
        for item in user_item_matrix.columns:
            if user_item_matrix.loc[user, item] == 0:
                if method == "hybrid":
                    pred = predict_hybrid(user, item, user_item_matrix, item_similarity, user_similarity, alpha=alpha, k=k)
                else:
                    pred = predict_cf(user, item, user_item_matrix, similarity_matrix,mode=method,k=k)

                predictions.append((user, item, pred))

    return pd.DataFrame(predictions, columns=["User", "Item", "Prediction"])

In [207]:
# -----------------------------
# RECOMENDAÇÃO
# -----------------------------
def recommend_from_other_user(target_user, other_user, user_item_matrix):

    print(f"\nRecomendando para usuário {target_user} com base no usuário {other_user}")

    # Itens que target_user já jogou
    target_items = user_item_matrix.loc[target_user]

    # Itens que other_user já jogou
    other_items = user_item_matrix.loc[other_user]

    # Pegar apenas itens que:
    # - other_user jogou (rating > 0)
    # - target_user NÃO jogou (rating == 0)
    candidate_items = other_items[
        (other_items > 0) & (target_items == 0)
    ]

    if len(candidate_items) == 0:
        print("Nenhum item novo para recomendar.")
        return

    # Ordenar pelos melhores ratings do other_user
    recommendations = candidate_items.sort_values(ascending=False)

    print("\nItens recomendados:")
    print(recommendations.head(5))


# ---------------- RECOMENDAÇÃO USER-BASED EXPLICÁVEL ---------------- #

def recommend_user_based_explain(target_user, user_item_matrix, user_similarity_df, top_n=5):

    print(f"\nRecomendações para usuário {target_user}")

    # Similaridade do usuário alvo com todos os outros
    similarities = user_similarity_df.loc[target_user].drop(target_user)
    # Pegar itens que o usuário ainda NÃO jogou
    user_ratings = user_item_matrix.loc[target_user]
    items_not_played = user_ratings[user_ratings == 0].index

    predictions = {}
    explanations = {}

    for item in items_not_played:
        # Usuários que avaliaram esse item
        users_who_rated = user_item_matrix[user_item_matrix[item] > 0].index

        numerator = 0
        denominator = 0
        contributors = []

        for other_user in users_who_rated:

            sim = similarities.get(other_user, 0)

            if sim > 0:# só similaridade positiva
                rating = user_item_matrix.loc[other_user, item]
                contribution = sim * rating

                numerator += contribution
                denominator += abs(sim)

                contributors.append({
                    "user": other_user,
                    "similarity": sim,
                    "rating": rating,
                    "contribution": contribution
                })

        if denominator > 0:
            predicted_rating = numerator / denominator
            predictions[item] = predicted_rating
            explanations[item] = contributors

    if len(predictions) == 0:
        print("Nenhum item possível para recomendar.")
        return

    recommendations = (
        pd.Series(predictions)
        .sort_values(ascending=False)
        .head(top_n)
    )

    # Mostrar explicação detalhada
    for item in recommendations.index:
        print("\n---------------------------------------")
        print(f"Explicação para {item}")
        print("---------------------------------------")

        contribs = sorted(
            explanations[item],
            key=lambda x: x["contribution"],
            reverse=True
        )

        for c in contribs[:5]:  # top 5 contribuidores
            print(
                f"Usuário {c['user']} | "
                f"similaridade={round(c['similarity'],3)} | "
                f"nota={round(c['rating'],2)} | "
                f"contribuição={round(c['contribution'],3)}"
            )
# Recommend items similar to 'Item1'
def recommend_similar_items(item, similarity_df, top_n=3):
  return similarity_df[item].sort_values(ascending=False)[1:top_n+1]

def print_user_ratings(user_item_matrix, user_similarity_df, user1_idx=0, user2_idx=1):
    # seleciona usuários
    u1 = user_similarity_df.index[user1_idx]
    u2 = user_similarity_df.index[user2_idx]

    print("Notas do usuário 1:")
    print(user_item_matrix.loc[u1].dropna())

    print("\nNotas do usuário 2:")
    print(user_item_matrix.loc[u2].dropna())

def recommend_hybrid(user,
                     user_item_matrix,
                     item_similarity_df,
                     user_similarity_df,
                     top_n=5,
                     alpha=0.5):

    items = user_item_matrix.columns
    predictions = {}

    for item in items:
        if user_item_matrix.loc[user, item] == 0:  # não avaliou
            predictions[item] = predict_hybrid(
                user,
                item,
                user_item_matrix,
                item_similarity_df,
                user_similarity_df,
                alpha
            )

    ranked = sorted(predictions.items(),
                    key=lambda x: x[1],
                    reverse=True)

    return ranked[:top_n]

In [208]:
def aggregate_recommendations_generic(
    recommended_levels_df,
    dungeon_metadata_df,
    user_profile_df,
    ratings_df,
    columns_to_use,
    metric_function,
    decision_function,
    top_n=5,
    scaler__=None
):
    """
    Função genérica para agregar recomendações.

    columns_to_use: lista de colunas do metadata
    metric_function: função que recebe user_recs e retorna dict de métricas
    decision_function: função que recebe rating_mean + métricas e retorna recomendação
    """

    available_cols = [col for col in columns_to_use if col in dungeon_metadata_df.columns]

    if len(available_cols) == 0:
        print("Nenhuma coluna encontrada")
        return pd.DataFrame()

    resultado = []

    for user in recommended_levels_df['User'].unique():

        # Top N recomendações
        user_recs = (
            recommended_levels_df[recommended_levels_df['User'] == user]
            .head(top_n)
            .copy()
        )

        if len(user_recs) == 0:
            continue

        # Merge metadata
        user_recs = user_recs.merge(
            dungeon_metadata_df[['LevelName'] + available_cols],
            left_on='Item',
            right_on='LevelName',
            how='left'
        )

        # Merge rating
        user_recs = user_recs.merge(
            ratings_df[ratings_df['User'] == user][['Item', 'Rating']],
            on='Item',
            how='left'
        )

        rating_mean = user_recs['Rating'].mean()

        # Perfil
        user_profile = user_profile_df[user_profile_df['PlayerId'] == user]
        preform = user_profile['PreFormAnswers'].iloc[0] if len(user_profile) > 0 else 'N/A'

        # Métricas específicas
        metrics = metric_function2(user_recs, scaler__)

        # Decisão específica
        recomendacao = decision_function2(rating_mean, metrics)

        resultado.append({
            'User': user,
            'PreFormAnswers': preform,
            'rating_medio': round(rating_mean, 2),
            **metrics,
            'recomendacao': recomendacao
        })

    return pd.DataFrame(resultado)

def aggregate_recommendations_generic(
    recommended_levels_df,
    dungeon_metadata_df,
    user_profile_df,
    ratings_df,
    columns_to_use,
    metric_function,
    decision_function,
    top_n=5,
    scaler__=None
):
    """
    Função genérica para agregar recomendações.

    columns_to_use: lista de colunas do metadata
    metric_function: função que recebe user_recs e retorna dict de métricas
    decision_function: função que recebe rating_mean + métricas e retorna recomendação
    """

    available_cols = [col for col in columns_to_use if col in dungeon_metadata_df.columns]

    if len(available_cols) == 0:
        print("Nenhuma coluna encontrada")
        return pd.DataFrame()

    resultado = []

    for user in recommended_levels_df['User'].unique():

        # Top N recomendações
        user_recs = (
            recommended_levels_df[recommended_levels_df['User'] == user]
            .head(top_n)
            .copy()
        )

        if len(user_recs) == 0:
            continue

        # Merge metadata
        user_recs = user_recs.merge(
            dungeon_metadata_df[['LevelName'] + available_cols],
            left_on='Item',
            right_on='LevelName',
            how='left'
        )

        # Merge rating
        user_recs = user_recs.merge(
            ratings_df[ratings_df['User'] == user][['Item', 'Rating']],
            on='Item',
            how='left'
        )

        rating_mean = user_recs['Rating'].mean()

        # Perfil
        user_profile = user_profile_df[user_profile_df['PlayerId'] == user]
        preform = user_profile['PreFormAnswers'].iloc[0] if len(user_profile) > 0 else 'N/A'

        # Métricas específicas
        metrics = metric_function(user_recs, scaler__)

        # Decisão específica
        recomendacao = decision_function(rating_mean, metrics)

        resultado.append({
            'User': user,
            'PreFormAnswers': preform,
            'rating_medio': round(rating_mean, 2),
            **metrics,
            'recomendacao': recomendacao
        })

    return pd.DataFrame(resultado)

def enemy_metrics(user_recs,scaler):
    kill_rate_mean = user_recs['EnemyKillRate'].mean()
    return {
        'kill_rate_medio': round(kill_rate_mean, 2)
    }

def enemy_decision(rating_mean, metrics):
    kill_rate = metrics['kill_rate_medio']

    if rating_mean >= 0.7:
        if kill_rate >= 0.7:
            return "MUITOS inimigos"
        elif kill_rate >= 0.5:
            return "MEDIA quantidade"
        else:
            return "POUCOS inimigos"
    else:
        return "MUITO POUCOS inimigos"

def quest_metrics(user_recs,scaler):
    quest_means = {}
    quest_columns = [
                    'CompletedCreativityQuests',
                    'CompletedKillQuests',
                    'CompletedExploreQuests',
                    'CompletedMasteryQuests',
                    'CompletedImmersionQuests',
                    'CompletedReportQuests',
                    'CompletedListenQuests',
                    'CompletedExchangeQuests',
                    'CompletedGiveQuests',
                    'CompletedGoToQuests',
                    'CompletedGatherQuests'
    ]

    quest_names = {
        'CompletedKillQuests': 'combate',
        'CompletedExploreQuests': 'exploracao',
        'CompletedCreativityQuests': 'criatividade',
        'CompletedMasteryQuests': 'maestria',
        'CompletedImmersionQuests': 'imersao',
        'CompletedReportQuests': 'report',
        'CompletedListenQuests': 'listen',
        'CompletedExchangeQuests': 'troca',
        'CompletedGiveQuests': 'doacao',
        'CompletedGoToQuests': 'locomocao',
        'CompletedGatherQuests': 'coleta'
    }



    for col in quest_columns:
        if col in user_recs.columns:
            quest_means[col] = user_recs[col].mean()

    if not quest_means:
        return {'preferencia': 'N/A'}

    sorted_quests = sorted(quest_means.items(), key=lambda x: x[1], reverse=True)

    top3 = []
    for quest, score in sorted_quests[:3]:
        nome = quest_names.get(quest, quest)
        top3.append(f"{nome}({score:.2f})")

    return {
        'top3_quests': top3,
        'quest_primaria': quest_names.get(sorted_quests[0][0], sorted_quests[0][0]),
        'score_primario': round(sorted_quests[0][1], 2)
    }

def quest_decision(rating_mean, metrics):
    return " > ".join(metrics['top3_quests']) if 'top3_quests' in metrics else "N/A"

def npc_metrics(user_recs, scaler):
    total = user_recs['TotalNpcs'].mean()
    interacted = user_recs['NpcsInteracted'].mean()
    ratio = interacted / total if total > 0 else 0

    return {
        'interacao_ratio': round(ratio, 2)
    }

def npc_decision(rating_mean, metrics):
    ratio = metrics['interacao_ratio']

    if rating_mean >= 0.7:
        if ratio >= 0.7:
            return "MUITOS NPCs"
        elif ratio >= 0.4:
            return "MEDIA quantidade"
        else:
            return "POUCOS NPCs"
    else:
        return "MUITO POUCOS NPCs"

def progress_metrics(user_recs,scaler):
    treasure_ratio = (
        (user_recs['TreasuresCollected'] / user_recs['TotalTreasure']).mean()
        if 'TotalTreasure' in user_recs.columns and 'TreasuresCollected' in user_recs.columns
        else 0
    )

    time_mean = (
        user_recs['TimeToFinish'].mean()
        if 'TimeToFinish' in user_recs.columns
        else 0
    )

    return {
        'treasure_ratio': round(treasure_ratio, 2),
        'tempo_medio': round(time_mean, 2)
    }


def progress_decision(rating_mean, metrics):
    ratio = metrics['treasure_ratio']

    if rating_mean >= 0.7:
        if ratio >= 0.7:
            return "MUITOS tesouros"
        elif ratio >= 0.4:
            return "MEDIA quantidade de tesouros"
        else:
            return "POUCOS tesouros"
    else:
        return "POUCOS tesouros"

'''def structure_metrics(user_recs, scaler=None):
    """
    Calcula métricas desnormalizando os valores
    """
    if scaler is not None:
        # Pegar os valores normalizados
        rooms_norm = user_recs['TotalRooms'].values.reshape(-1, 1)

        # Desnormalizar
        rooms_real = scaler.inverse_transform(rooms_norm)
        rooms_mean = rooms_real.mean()

        # Fazer o mesmo para outras colunas
        keys_norm = user_recs['TotalKeys'].values.reshape(-1, 1)
        keys_real = scaler.inverse_transform(keys_norm)
        keys_mean = keys_real.mean()

        locks_norm = user_recs['TotalLocks'].values.reshape(-1, 1)
        locks_real = scaler.inverse_transform(locks_norm)
        locks_mean = locks_real.mean()

        revisit_ratio = (
            (user_recs['RoomsEntered'] / user_recs['UniqueRoomsEntered'])#.mean()
            #if 'RoomsEntered' in user_recs.columns and 'UniqueRoomsEntered' in user_recs.columns
            #else 0
        )

        return {
            'rooms_medio': round(rooms_mean),  # Agora é inteiro!
            'keys_medio': round(keys_mean),
            'locks_medio': round(locks_mean),
            'revisit_ratio': round(revisit_ratio, 2)
        }
    else:
        return'''
def structure_metrics(user_recs, scaler__):
    #AQUIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
    rooms_mean = user_recs['TotalRooms'].mean() if 'TotalRooms' in user_recs.columns else 0
    keys_mean = user_recs['TotalKeys'].mean() if 'TotalKeys' in user_recs.columns else 0
    locks_mean = user_recs['TotalLocks'].mean() if 'TotalLocks' in user_recs.columns else 0

    revisit_ratio = (
        (user_recs['RoomsEntered'] / user_recs['UniqueRoomsEntered'])#.mean()
        #if 'RoomsEntered' in user_recs.columns and 'UniqueRoomsEntered' in user_recs.columns
        #else 0
    )

    return {
        'rooms_medio': round(rooms_mean, 1),
        'keys_medio': round(keys_mean, 1),
        'locks_medio': round(locks_mean, 1),
        'revisit_ratio': round(revisit_ratio, 2)
    }


def structure_decision(rating_mean, metrics):#aquiiiiiii?
    rooms = metrics['rooms_medio']

    if rating_mean >= 0.7:
        if rooms >= 20:
            return "DUNGEONS GRANDES"
        elif rooms >= 10:
            return "DUNGEONS MEDIAS"
        else:
            return "DUNGEONS PEQUENAS"
    else:
        return "DUNGEONS PEQUENAS"

def keys_metrics(user_recs):
    media = user_recs['TotalKeys'].mean()

    return {
        'media_chaves': round(media, 1),
        'min_chaves': user_recs['TotalKeys'].min(),
        'max_chaves': user_recs['TotalKeys'].max()
    }


def keys_decision(rating_mean, metrics):
    media = metrics['media_chaves']

    if media <= 3:
        return "0-3 chaves"
    elif media <= 6:
        return "4-6 chaves"
    elif media <= 9:
        return "7-9 chaves"
    else:
        return "10+ chaves"

In [209]:
# -----------------------------
# ANÁLISES
# -----------------------------
def inspect_dataset(user_frame, dungeon_frame):
    profiles = user_frame['PreFormAnswers'].apply(tuple)

    print("Combinações únicas:")
    print(sorted(profiles.unique()))

    print("\nQuantidade de combinações únicas:")
    print(profiles.nunique())

    print("\nFrequência de cada combinação:")
    print(profiles.value_counts())

    dungeon_frame = pd.merge(
        dungeon_frame,
        user_frame[['PlayerId', 'PreFormAnswers']],
        on=['PlayerId'],
        how='left'
    )

    print("\nFrequência no dungeon_frame final:")
    print(dungeon_frame['PreFormAnswers'].value_counts())

    # Listar todas as colunas
    colunas = dungeon_frame.columns.tolist()

    print(f"\nTotal de colunas: {len(colunas)}\n")

    # Exibir em ordem alfabética numerada
    for i, col in enumerate(sorted(colunas), 1):
        print(f"{i:3d}. {col}")

def analyze_user_item_matrix(user_item_matrix):

    print("------------- VERIFICAÇÃO DOS DADOS -------------")
    print("Número de usuários:", user_item_matrix.shape[0])
    print("Número de itens:", user_item_matrix.shape[1])

    print("\nQuantidade média de avaliações por usuário:")
    print(user_item_matrix.count(axis=1).mean())

    print("\nQuantidade média de usuários por item:")
    print(user_item_matrix.count(axis=0).mean())

    print("\n------------- VARIÂNCIA ITEM POR ITEM -------------")
    print("Variância de cada item:")
    print(user_item_matrix.var())

In [210]:
# -----------------------------
# CÁLCULOS
# -----------------------------
def evaluate_rating_metrics(predictions_df, test_matrix):

    y_true = []
    y_pred = []

    for _, row in predictions_df.iterrows():
        user = row["User"]
        item = row["Item"]

        if user in test_matrix.index and item in test_matrix.columns:
            true_rating = test_matrix.loc[user, item]
            if true_rating != 0:
                y_true.append(true_rating)
                y_pred.append(row["Prediction"])

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return rmse, mae

def compute_rmse(test_df, prediction_col, rating_col):
    """
    Calcula RMSE ignorando:
        - valores NaN
        - casos sem predição
        - divisão por zero
    """

    # Manter apenas linhas com valor real e predito
    mask = (
        test_df[rating_col].notna() &
        test_df[prediction_col].notna()
    )

    y_true = test_df.loc[mask, rating_col].values
    y_pred = test_df.loc[mask, prediction_col].values

    if len(y_true) == 0:
        raise ValueError("Nenhuma predição válida para calcular RMSE.")

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    return rmse

def precision_at_k(rec_df, test_df, k=5, threshold=1.0):
    precision_list = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[(test_df["User"] == user) & (test_df["Rating"] >= threshold)]["Item"].values

        if len(true_items) == 0:
            continue

        hits = len(set(top_k) & set(true_items))
        precision_list.append(hits / k)

    return np.mean(precision_list)
def recall_at_k(rec_df, test_df, k=5, threshold=1.0):
    recall_list = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[(test_df["User"] == user) & (test_df["Rating"] >= threshold)]["Item"].values

        if len(true_items) == 0:
            continue

        hits = len(set(top_k) & set(true_items))
        recall_list.append(hits / len(true_items))

    return np.mean(recall_list)
def hit_rate_at_k(rec_df, test_df, k=5):
    hits = 0
    users = test_df["User"].unique()

    for user in users:
        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[test_df["User"] == user]["Item"].values

        if len(set(top_k) & set(true_items)) > 0:
            hits += 1

    return hits / len(users)
def mrr(rec_df, test_df):
    rr_total = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].reset_index()

        true_items = set(test_df[test_df["User"] == user]["Item"])

        rank = 0

        for i, row in top_k.iterrows():
            if row["Item"] in true_items:
                rank = i + 1
                break

        if rank > 0:
            rr_total.append(1 / rank)

    return np.mean(rr_total)

def similiarity_pearson(user_item_matrix):
    # Pearson item-item
    item_similarity_pearson = user_item_matrix.corr(method='pearson')

    print("\nITEM POR ITEM (PEARSON):\n")
    print(item_similarity_pearson)
    item_similarity_df = item_similarity_pearson

    user_similarity_pearson = user_item_matrix.T.corr(method='pearson')
    user_similarity_pearson = user_similarity_pearson.fillna(0)

    print("\nUSUARIO POR USUARIO (PEARSON):\n")
    print(user_similarity_pearson)

    user_similarity_df  = user_similarity_pearson

    return item_similarity_df, user_similarity_df

def similiarity_cosine(user_item_matrix):
    # Compute item similarity using cosine similarity
    item_similarity = cosine_similarity(user_item_matrix.T)
    item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

    #centralizando
    user_item_centered = user_item_matrix.sub(user_item_matrix.mean(axis=1), axis=0)
    #print("\nconsseno cetnralizado:\n")
    #print(user_item_centered)

    # Similaridade entre itens
    # item por item
    item_similarity_centered = cosine_similarity(user_item_centered.T)
    item_similarity_centered_df = pd.DataFrame(item_similarity_centered, index=user_item_centered.columns, columns=user_item_centered.columns)

    #print("\nconsseno:\n")
    #print(item_similarity_df)

    #print("\nconsseno cetnralizado:\n")
    #print(item_similarity_centered)

    # usuario por usuário
    user_similarity = cosine_similarity(user_item_centered)
    user_similarity_df = pd.DataFrame( user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index )
    #print("\nUSUARIO POR USUARIO:\n")
    #print(user_similarity_df.iloc[:50, :50])

    return item_similarity_df, user_similarity_df

def compute_performance_rating(df):
    df['TotalLostHealth_norm'] = minmax(df['TotalLostHealth'])

    return (
        df['Success Rate']
        - df['TotalLostHealth_norm']
        + df['EnemyKillRate']
        + df['MapCompletion']
    )

def build_item_from_structure(df):
    df['TotalRooms'] = df['TotalRooms'].replace(0, 1)

    return (
        "R" + df['TotalRooms'].astype(str) +
        "_K" + df['TotalKeys'].astype(str))# +        "_RR" + (df['RoomsEntered'] - df['UniqueRoomsEntered']).astype(str)    )

def compute_hybrid_rating(df, alpha=0.5):
    df_valid = df['PostFormAnswers'].apply(mean_post_forms)
    df_comp = compute_performance_rating(df)

    return alpha * df_comp + (1 - alpha) * df_valid

def mean_post_forms(x):
    return sum(x)/len(x) if isinstance(x, list) else x

def minmax(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

In [211]:
def recomendando(method, user_item_matrix, user_similarity_df, item_similarity_df,_df):
  # ------------- RECOMENDAR ITEM POR ITEM ------------#
      # esta comentado, pq nao fareirpos oturos casos
      if(method >=1 and method <=4):
        # Example recommendation
        similar_items = recommend_similar_items('A=4, C=3, I=2, M=1', item_similarity_df)
        print("\nItems similar to A=4, C=3, I=2, M=1:", similar_items)

      # Recomendar todos os itens por similarida itens
      print("--------------------6----------------------------")
      all_recommendations = {}

      for item in user_item_matrix.columns:
          all_recommendations[item] = recommend_similar_items(item, item_similarity_df)

      print(all_recommendations)

      # ------------- SIMILIARIDA USUARIO POR USUARIO ------------#
      print("--------------------7----------------------------")

      # print abaixo é pra ENTENDNEO O RESULTADO ACIMA
      print_user_ratings(user_item_matrix, user_similarity_df)

      # ------------- testando um usuario com mais de um nivel ------------#
      print("--------------------8----------------------------")
      #analyze_user_activity(user_item_matrix) # -> TODO; Descobri onde que tivrei e voltar

      # Recomendar para u1 baseado em u2
      print("--------------------9----------------------------")
      recommend_from_other_user(user_item_matrix.index[0], user_item_matrix.index[1], user_item_matrix)

      # Recomendar para u2 baseado em u1
      print("--------------------10----------------------------")
      recommend_from_other_user(user_item_matrix.index[1], user_item_matrix.index[0], user_item_matrix)

      # Escolher usuário
      print("--------------------13----------------------------")
      recommend_user_based_explain(user_item_matrix.index[0], user_item_matrix, user_similarity_df, top_n=5)

      # 🔥 MULTI-VIEW RECOMMENDATION (NOVO)
      print("--------------------14----------------------------")
      if method == 13:
          print("\n\n=========== MULTI VIEW ===========")
      elif (method >= 14):
          print("\n=========== HYBRID RECOMMENDATION ===========")
          recommend_hybrid(user_item_matrix.index[0], user_item_matrix, item_similarity_df, user_similarity_df)

In [212]:
def run_aggregation(
    title,
    aggregation_function,
    rec_df,
    _df,
    user_profiles,
    collab_df,
    columns_to_print,
    extra_param=None,
    top_n=5,
    scaler__=None
):

    if extra_param is not None:
        result = aggregation_function(
            rec_df,
            extra_param,
            user_profiles,
            collab_df,
            top_n=top_n
        )
    else:
        result = aggregation_function(
            rec_df,
            _df,
            user_profiles,
            collab_df,
            top_n=top_n
        )

    print(f"\n=== {title} ===")

    if len(result) > 0:
        print(result[columns_to_print].to_string())

    return result

def generate_recommendations(
    test_df,
    all_items,
    user_item_matrix,
    similarity_matrix,
    predict_function,
    method="user",
    alpha=0.5,
    k=20
):
    recommendations = []

    for user in test_df["User"].unique():
        for item in all_items:

            score = predict_cf(user, item, user_item_matrix, similarity_matrix,mode=method,k=k)

            recommendations.append({
                "User": user,
                "Item": item,
                "Score": score
            })

    # Converte para DataFrame
    rec_df = pd.DataFrame(recommendations)

    # Ordena
    rec_df = rec_df.sort_values(
        ["User", "Score"],
        ascending=[True, False]
    )

    return rec_df

def evaluate_error_metrics(pred_df, label=""):

    pred_df = pred_df.dropna()

    rmse = np.sqrt(mean_squared_error(
        pred_df["Rating"],
        pred_df["Prediction"]
    ))

    mae = mean_absolute_error(
        pred_df["Rating"],
        pred_df["Prediction"]
    )

    print(f"\n===== {label} ERROR METRICS =====")
    print("Predições válidas:", len(pred_df))
    print("RMSE:", rmse)
    print("MAE:", mae)

    return rmse, mae

def evaluate_ranking_metrics(rec_df, test_df, k=8, threshold=0.0, label=""):

    precision = precision_at_k(rec_df, test_df, k=k, threshold=threshold)
    recall    = recall_at_k(rec_df, test_df, k=k, threshold=threshold)

    hr  = hit_rate_at_k(rec_df, test_df, k=k)
    mrr_score = mrr(rec_df, test_df)

    print(f"\n===== {label} RANKING METRICS =====")
    print(f"Precision@{k}:", precision)
    print(f"Recall@{k}:", recall)
    print(f"HitRate@{k}:", hr)
    print("MRR:", mrr_score)

    return precision, recall, hr, mrr_score

def debug_predictions(pred_df, test_df):

    print("\n===== DEBUG INTERSEÇÃO =====")
    print("Itens únicos preditos:", pred_df['Item'].nunique())
    print("Itens únicos no teste:", test_df['Item'].nunique())

    intersection = set(pred_df['Item']) & set(test_df['Item'])
    print("Interseção de itens:", len(intersection))
    print("Itens em comum:", intersection)

    print("\n===== DEBUG USUÁRIOS =====")
    print(test_df.groupby("User")["Item"].count())

def evaluate_model(
    pred_df,
    rec_df,
    test_df,
    label="MODEL",
    k=8,
    threshold=0.0,
    debug=False
):

    print(f"\n==============================")
    print(f"===== {label} =====")
    print(f"==============================")

    if debug:
        debug_predictions(pred_df, test_df)

    evaluate_error_metrics(pred_df, label)
    evaluate_ranking_metrics(rec_df, test_df, k, threshold, label)

def recomenda_tabela2(_df, collab_df, rec_user_df, scaler__):
    quest_columns = [
        'CompletedCreativityQuests',
        'CompletedKillQuests',
        'CompletedExploreQuests',
        'CompletedMasteryQuests',
        'CompletedImmersionQuests',
        'CompletedReportQuests',
        'CompletedListenQuests',
        'CompletedExchangeQuests',
        'CompletedGiveQuests',
        'CompletedGoToQuests',
        'CompletedGatherQuests'
    ]

    available_quests = [col for col in quest_columns if col in _df.columns]
    dungeon_quests = _df[['LevelName'] + available_quests].drop_duplicates()
    user_profiles = _df[['PlayerId', 'PreFormAnswers']].drop_duplicates()

    # Executar agregações (mantenha como está)...
    agregacao_quests = run_aggregation("RECOMENDACOES AGREGADAS POR PREFERENCIA DE QUESTS",
        lambda rec_df, meta_df, user_profiles, collab_df, top_n:
         aggregate_recommendations_generic(
             rec_df,
             meta_df,
             user_profiles,
             collab_df,
             columns_to_use=quest_columns,
             metric_function=quest_metrics,
             decision_function=quest_decision,
             top_n=top_n
         ), rec_user_df, _df, user_profiles,
       collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
       extra_param=dungeon_quests,
       scaler__=scaler__
    )

    agregacao_inimigos = run_aggregation("RECOMENDACOES AGREGADAS POR DIFICULDADE DE INIMIGOS",
        lambda rec_df, meta_df, user_profiles, collab_df, top_n:
         aggregate_recommendations_generic(
             rec_df,
             meta_df,
             user_profiles,
             collab_df,
             columns_to_use=['TotalEnemies'],  # <-- VALOR REAL
             metric_function=enemy_metrics2,
             decision_function=enemy_decision2,
             top_n=top_n,
             scaler__=scaler__
         ), rec_user_df, _df, user_profiles,
       collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
       scaler__=scaler__
    )

    agregacao_npcs = run_aggregation("RECOMENDACOES AGREGADAS POR NPCS",
        lambda rec_df, meta_df, user_profiles, collab_df, top_n:
         aggregate_recommendations_generic(
             rec_df,
             meta_df,
             user_profiles,
             collab_df,
             columns_to_use=['TotalNpcs'],  # <-- VALOR REAL
             metric_function=npc_metrics2,
             decision_function=npc_decision2,
             top_n=top_n,
             scaler__=scaler__
         ), rec_user_df, _df, user_profiles,
       collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
       scaler__=scaler__
    )

    agregacao_progresso = run_aggregation("RECOMENDACOES AGREGADAS POR PROGRESSO (TESOUROS/TEMPO)",
        lambda rec_df, meta_df, user_profiles, collab_df, top_n:
         aggregate_recommendations_generic(
             rec_df,
             meta_df,
             user_profiles,
             collab_df,
             columns_to_use=['TotalTreasure'],  # <-- VALOR REAL
             metric_function=progress_metrics2,
             decision_function=progress_decision2,
             top_n=top_n,
             scaler__=scaler__
         ), rec_user_df, _df, user_profiles,
       collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
       scaler__=scaler__
    )

    agregacao_estrutura = run_aggregation("RECOMENDACOES AGREGADAS POR ESTRUTURA (SALAS)",
        lambda rec_df, meta_df, user_profiles, collab_df, top_n:
         aggregate_recommendations_generic(
             rec_df,
             meta_df,
             user_profiles,
             collab_df,
             columns_to_use=['TotalRooms'],  # <-- VALOR REAL
             metric_function=structure_metrics2,
             decision_function=structure_decision2,
             top_n=top_n,
             scaler__=scaler__
         ), rec_user_df, _df, user_profiles,
       collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
       scaler__=scaler__
    )

    # DataFrame com os VALORES REAIS
    df_valores_reais = pd.DataFrame({
        'User': agregacao_quests['User'],
        'PreFormAnswers': agregacao_quests['PreFormAnswers'],
        'rating_medio': agregacao_quests['rating_medio'],

        # VALORES REAIS (não normalizados)
        'media_inimigos': agregacao_inimigos['total_enemies_medio'],  # você precisa ajustar a métrica
        'media_npcs': agregacao_npcs['total_npcs_medio'],
        'media_tesouros': agregacao_progresso['total_treasure_medio'],
        'media_salas': agregacao_estrutura['total_rooms_medio'],

        # Dados das quests
        'quest_primaria': agregacao_quests['quest_primaria'],
        'score_primario': agregacao_quests['score_primario'],
    })

    # Exibir os valores reais
    print("\n=== VALORES REAIS DAS MÉTRICAS 22222 ===")
    print(df_valores_reais[['User', 'media_inimigos', 'media_npcs', 'media_tesouros', 'media_salas']])

def enemy_metrics2(user_recs, scaler):
    # Retorna a MÉDIA REAL de inimigos
    return {
        'total_enemies_medio': user_recs['TotalEnemies'].mean() if 'TotalEnemies' in user_recs.columns else 0
    }

def npc_metrics2(user_recs, scaler):
    # Retorna a MÉDIA REAL de NPCs
    return {
        'total_npcs_medio': user_recs['TotalNpcs'].mean() if 'TotalNpcs' in user_recs.columns else 0
    }

def progress_metrics2(user_recs, scaler):
    # Retorna a MÉDIA REAL de tesouros
    return {
        'total_treasure_medio': user_recs['TotalTreasure'].mean() if 'TotalTreasure' in user_recs.columns else 0
    }

def structure_metrics2(user_recs, scaler__):
    # Retorna a MÉDIA REAL de salas
    return {
        'total_rooms_medio': user_recs['TotalRooms'].mean() if 'TotalRooms' in user_recs.columns else 0
    }
def enemy_decision2(rating_mean, metrics):
    total_enemies = metrics['total_enemies_medio']

    if rating_mean >= 0.7:
        if total_enemies >= 30:
            return f"MUITOS inimigos ({total_enemies:.1f})"
        elif total_enemies >= 15:
            return f"MEDIA quantidade ({total_enemies:.1f})"
        else:
            return f"POUCOS inimigos ({total_enemies:.1f})"
    else:
        return f"MUITO POUCOS inimigos ({total_enemies:.1f})"

def npc_decision2(rating_mean, metrics):
    total_npcs = metrics['total_npcs_medio']

    if rating_mean >= 0.7:
        if total_npcs >= 20:
            return f"MUITOS NPCs ({total_npcs:.1f})"
        elif total_npcs >= 10:
            return f"MEDIA quantidade ({total_npcs:.1f})"
        else:
            return f"POUCOS NPCs ({total_npcs:.1f})"
    else:
        return f"MUITO POUCOS NPCs ({total_npcs:.1f})"

def progress_decision2(rating_mean, metrics):
    total_treasure = metrics['total_treasure_medio']

    if rating_mean >= 0.7:
        if total_treasure >= 50:
            return f"MUITOS tesouros ({total_treasure:.1f})"
        elif total_treasure >= 25:
            return f"MEDIA quantidade de tesouros ({total_treasure:.1f})"
        else:
            return f"POUCOS tesouros ({total_treasure:.1f})"
    else:
        return f"POUCOS tesouros ({total_treasure:.1f})"

def structure_decision2(rating_mean, metrics):
    total_rooms = metrics['total_rooms_medio']

    if rating_mean >= 0.7:
        if total_rooms >= 20:
            return f"DUNGEONS GRANDES ({total_rooms:.1f} salas)"
        elif total_rooms >= 10:
            return f"DUNGEONS MEDIAS ({total_rooms:.1f} salas)"
        else:
            return f"DUNGEONS PEQUENAS ({total_rooms:.1f} salas)"
    else:
        return f"DUNGEONS PEQUENAS ({total_rooms:.1f} salas)"

def recomenda_tabela(_df, collab_df, rec_user_df, scaler__):
  quest_columns = [
                    'CompletedCreativityQuests',
                    'CompletedKillQuests',
                    'CompletedExploreQuests',
                    'CompletedMasteryQuests',
                    'CompletedImmersionQuests',
                    'CompletedReportQuests',
                    'CompletedListenQuests',
                    'CompletedExchangeQuests',
                    'CompletedGiveQuests',
                    'CompletedGoToQuests',
                    'CompletedGatherQuests'
  ]

  available_quests = [col for col in quest_columns if col in _df.columns]
  dungeon_quests = _df[['LevelName'] + available_quests].drop_duplicates()
  user_profiles = _df[['PlayerId', 'PreFormAnswers']].drop_duplicates()

  agregacao_quests = run_aggregation("RECOMENDACOES AGREGADAS POR PREFERENCIA DE QUESTS",
       lambda rec_df, meta_df, user_profiles, collab_df, top_n:
        aggregate_recommendations_generic(
            rec_df,
            meta_df,
            user_profiles,
            collab_df,
            columns_to_use=quest_columns,
            metric_function=quest_metrics,
            decision_function=quest_decision,
            top_n=top_n
        ), rec_user_df, _df, user_profiles,
      collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
      extra_param=dungeon_quests,
      scaler__=scaler__
  )
  # Converter para lista
  #print("bbbbbbbbbbbbbb")
  #lista_recomendacoes = agregacao_quests['recomendacao'].tolist()
  #print(lista_recomendacoes)

  ## Iterar sobre as recomendações
  #for rec in agregacao_quests['recomendacao']:
      #print(rec)
  agregacao_inimigos = run_aggregation("RECOMENDACOES AGREGADAS POR DIFICULDADE DE INIMIGOS",
      lambda rec_df, meta_df, user_profiles, collab_df, top_n:
        aggregate_recommendations_generic(
            rec_df,
            meta_df,
            user_profiles,
            collab_df,
            columns_to_use=['EnemyKillRate'],
            metric_function=enemy_metrics,
            decision_function=enemy_decision,
            top_n=top_n
        ), rec_user_df, _df, user_profiles,
      collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'kill_rate_medio', 'recomendacao'],
      scaler__=scaler__
  )

  agregacao_npcs = run_aggregation("RECOMENDACOES AGREGADAS POR NPCS",
      lambda rec_df, meta_df, user_profiles, collab_df, top_n:
        aggregate_recommendations_generic(
            rec_df,
            meta_df,
            user_profiles,
            collab_df,
            columns_to_use=['TotalNpcs','NpcsInteracted'],
            metric_function=npc_metrics,
            decision_function=npc_decision,
            top_n=top_n
        ), rec_user_df,  _df, user_profiles,
      collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'interacao_ratio', 'recomendacao'],
      scaler__=scaler__
  )

  agregacao_progresso = run_aggregation("RECOMENDACOES AGREGADAS POR PROGRESSO (TESOUROS/TEMPO)",
       lambda rec_df, meta_df, user_profiles, collab_df, top_n:
        aggregate_recommendations_generic(
            rec_df,
            meta_df,
            user_profiles,
            collab_df,
            columns_to_use=['TotalTreasure','TreasuresCollected','TimeToFinish'],
            metric_function=progress_metrics,
            decision_function=progress_decision,
            top_n=top_n
        ), rec_user_df, _df, user_profiles,
      collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'treasure_ratio', 'recomendacao'],
      scaler__=scaler__
  )

  agregacao_estrutura = run_aggregation("RECOMENDACOES AGREGADAS POR ESTRUTURA (SALAS/CHAVES/FECHADURAS)",
      lambda rec_df, meta_df, user_profiles, collab_df, top_n:
        aggregate_recommendations_generic(
            rec_df,
            meta_df,
            user_profiles,
            collab_df,
            columns_to_use=['TotalRooms','TotalKeys','TotalLocks','UniqueRoomsEntered','RoomsEntered'],
            metric_function=structure_metrics,
            decision_function=structure_decision,
            top_n=top_n
        ), rec_user_df,  _df, user_profiles,
      collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'rooms_medio', 'recomendacao'],
      scaler__=scaler__
  )

  # Criar DataFrame com todas as recomendações
  df_todas_recomendacoes = pd.DataFrame({
      'User': agregacao_quests['User'],
      'PreFormAnswers': agregacao_quests['PreFormAnswers'],
      'rating_medio': agregacao_quests['rating_medio'],
      'recomendacao_quests': agregacao_quests['recomendacao'],
      'recomendacao_inimigos': agregacao_inimigos['recomendacao'],
      'recomendacao_npcs': agregacao_npcs['recomendacao'],
      'recomendacao_progresso': agregacao_progresso['recomendacao'],
      'recomendacao_estrutura': agregacao_estrutura['recomendacao']
  })

  # Exibir os resultados
  print("------------------Exibir os resultados-------------------")
  print(df_todas_recomendacoes )
  print(f"\nTotal de usuários: {len(df_todas_recomendacoes )}")
  print(f"Colunas: {df_todas_recomendacoes .columns.tolist()}")

  # Depois de criar todas as agregações, acesse as colunas com valores reais:

  # Para QUESTs - os valores reais estão em 'top3_quests', 'quest_primaria', 'score_primario'
  valores_reais_quests = agregacao_quests[['User', 'top3_quests', 'quest_primaria', 'score_primario']]

  # Para INIMIGOS - o valor real está em 'kill_rate_medio'
  valores_reais_inimigos = agregacao_inimigos[['User', 'kill_rate_medio']]

  # Para NPCs - o valor real está em 'interacao_ratio'
  valores_reais_npcs = agregacao_npcs[['User', 'interacao_ratio']]

  # Para PROGRESSO - o valor real está em 'treasure_ratio'
  valores_reais_progresso = agregacao_progresso[['User', 'treasure_ratio']]

  # Para ESTRUTURA - o valor real está em 'rooms_medio'
  valores_reais_estrutura = agregacao_estrutura[['User', 'rooms_medio']]

  # DataFrame com todos os valores numéricos reais
  df_valores_reais = pd.DataFrame({
      'User': agregacao_quests['User'],
      'PreFormAnswers': agregacao_quests['PreFormAnswers'],
      'rating_medio': agregacao_quests['rating_medio'],

      # Valores reais das métricas
      'kill_rate_medio': agregacao_inimigos['kill_rate_medio'],
      'interacao_ratio': agregacao_npcs['interacao_ratio'],
      'treasure_ratio': agregacao_progresso['treasure_ratio'],
      'rooms_medio': agregacao_estrutura['rooms_medio'],

      # Dados das quests (valores reais)
      'quest_primaria': agregacao_quests['quest_primaria'],
      'score_primario': agregacao_quests['score_primario'],
      'top3_quests': agregacao_quests['top3_quests'],

      # Decisões textuais (se ainda quiser manter)
      'decisao_quests': agregacao_quests['recomendacao'],
      'decisao_inimigos': agregacao_inimigos['recomendacao'],
      'decisao_npcs': agregacao_npcs['recomendacao'],
      'decisao_progresso': agregacao_progresso['recomendacao'],
      'decisao_estrutura': agregacao_estrutura['recomendacao']
  })

  print(df_valores_reais)

  # Mostrar apenas valores numéricos reais
  print("\n=== VALORES REAIS DAS MÉTRICAS ===")
  print(df_valores_reais[['User', 'rating_medio', 'kill_rate_medio', 'interacao_ratio', 'treasure_ratio', 'rooms_medio']])

  # Mostrar com as quests
  print("\n=== VALORES REAIS COM QUESTS ===")
  print(df_valores_reais[['User', 'quest_primaria', 'score_primario', 'top3_quests']])

  # Estatísticas dos valores reais
  print("\n=== ESTATÍSTICAS DOS VALORES REAIS ===")
  print(df_valores_reais[['rating_medio', 'kill_rate_medio', 'interacao_ratio', 'treasure_ratio', 'rooms_medio']].describe())

  # Correlação entre as métricas
  print("\n=== CORRELAÇÃO ENTRE MÉTRICAS ===")
  print(df_valores_reais[['rating_medio', 'kill_rate_medio', 'interacao_ratio', 'treasure_ratio', 'rooms_medio']].corr())


def desnormalizar_em_outra_funcao(valor_normalizado, coluna, scaler, numeric_cols):
    """
    Desnormaliza um valor usando o scaler
    """
    idx = numeric_cols.index(coluna)
    minimo = scaler.data_min_[idx]
    maximo = scaler.data_max_[idx]

    valor_real = minimo + (valor_normalizado * (maximo - minimo))
    return round(valor_real)

def predicao(test_matrix, user_item_matrix, collab_df, user_similarity_df, item_similarity_df, method, _df, test_df, scaler__):
  # 1. Garantir interseção estrutural
      print("TEST MATRIX SHAPE BEFORE:", test_matrix.shape)
      print("TRAIN MATRIX SHAPE:", user_item_matrix.shape)

      common_users = set(collab_df["User"]).intersection(test_df["User"])
      common_items = set(collab_df["Item"]).intersection(test_df["Item"])

      print("\nANTES DO FILTRO:")
      print("Test size:", len(test_df))
      print("Common users:", len(common_users))
      print("Common items:", len(common_items))

      test_df = test_df[
          test_df["User"].isin(common_users) &
          test_df["Item"].isin(common_items)
      ]

      print("\nDEPOIS DO FILTRO:")
      print("Test size:", len(test_df))

      if len(test_df) == 0:
          print("\n TEST DF VIROU VAZIO → pulando métricas")
          return


      print("TEST MATRIX SHAPE AFTER:", test_matrix.shape)


      all_items = user_item_matrix.columns

      test_df,

      rec_user_df = generate_recommendations(
          test_df,
          all_items,
          user_item_matrix,
          user_similarity_df,
          predict_cf
      )

      rec_item_df = generate_recommendations(
          test_df,
          all_items,
          user_item_matrix,
          item_similarity_df,
          predict_cf
      )

      predictions = []
      predictions_item = []

      for _, row in test_df.iterrows():

          user = row["User"]
          item = row["Item"]
          true_rating = row["Rating"]

          # ---------- USER-BASED ----------
          pred_user = predict_cf(
              user,
              item,
              user_item_matrix,
              user_similarity_df
          )

          # ---------- ITEM-BASED ----------
          pred_item = predict_cf(
              user,
              item,
              user_item_matrix,
              item_similarity_df
          )

          predictions.append({
              "User": user,
              "Item": item,
              "Rating": true_rating,
              "Prediction": pred_user
          })

          predictions_item.append({
              "User": user,
              "Item": item,
              "Rating": true_rating,
              "Prediction": pred_item
          })


      print("\n===== ITEM-BASED =====")

      pred_df = pd.DataFrame(predictions)
      pred_item_df = pd.DataFrame(predictions_item)

      evaluate_model(
          pred_df,
          rec_user_df,
          test_df,
          label="USER-BASED",
          k=8,
          threshold=0.0,
          debug=True
      )

      evaluate_model(
          pred_item_df,
          rec_item_df,
          test_df,
          label="ITEM-BASED",
          k=8,
          threshold=0.0,
          debug=False
      )


      if method >= 20 and method <= 23:
            if len(rec_user_df) > 0:
                recomenda_tabela(_df, collab_df, rec_user_df, scaler__)
                run_combined_recommendation(
                    _df,
                    collab_df,
                    rec_user_df,
                    dungeon_metadata_df=_df,  # ou passe o dataframe correto com metadados
                    top_n=5
                )
                print("=-----------------------------*********************************************")
                recomenda_tabela2(_df, collab_df, rec_user_df, scaler__)
                run_combined_recommendation(
                    _df,
                    collab_df,
                    rec_user_df,
                    dungeon_metadata_df=_df,  # ou passe o dataframe correto com metadados
                    top_n=5
                )


def collaborative_filtering(_df, method, teste, recomenda, scaler__):
      print("-------------------- VISUALIZAÇÃO DOS DADOS ----------------------------")
      print(_df.shape)
      print(_df.head())
      print("-"*80)

      print("\nFrequência no dungeon_frame final:")
      print(_df['PreFormAnswers'].value_counts())

      # Número de usuários únicos pelo PlayerId
      num_unicos = _df['PlayerId'].nunique()
      print(f"\nNúmero de usuários únicos: {num_unicos}")

      print("-------------------- NORMALIZAÇÃO DOS DADOS ----------------------------")
      collab_df = build_collab_df(_df, method)
      test_df = build_collab_df(teste, method)

      print("\nMatriz para filtragem 10 :")
      print(test_df.head())

      print("\nMatriz para filtragem 1 :")
      print(collab_df.head())

      print("-------------------- CÁLCULOS ----------------------------")

      # Create user-item matrix
      user_item_matrix = collab_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)
      test_matrix = test_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)
      print(test_matrix.head())

      n_users = user_item_matrix.shape[0]
      n_items = user_item_matrix.shape[1]

      total_cells = n_users * n_items
      non_zero = (user_item_matrix > 0).sum().sum()

      sparsity = 1 - (non_zero / total_cells)

      print("\n===== SPARSITY =====")
      print("Users:", n_users)
      print("Items:", n_items)
      print("Total cells:", total_cells)
      print("Non-zero interactions:", non_zero)
      print("Sparsity:", sparsity)

      print("-------------------- VERIFICAÇÃO DOS DADOS ----------------------------")
      analyze_user_item_matrix(user_item_matrix)

      print("-------------------- SIMILIARIDADE ----------------------------")
      if(method % 2 == 0):
        item_similarity_df, user_similarity_df = similiarity_pearson(user_item_matrix)
      else:
        item_similarity_df, user_similarity_df = similiarity_cosine(user_item_matrix)

      if(recomenda):
        recomendando(method, user_item_matrix, user_similarity_df, item_similarity_df, _df)


      print("-------------------- PREDIÇÃO ----------------------------")
      predicao(test_matrix, user_item_matrix, collab_df, user_similarity_df, item_similarity_df, method, _df, test_df, scaler__)




def run(_df, method, recomenda):
    # Organizando o dataset
    _df['PreFormAnswers'] = _df['PreFormAnswers'].apply(lambda x: 'A=' + str(x[0]) + ", C=" + str(x[1]) + ", I=" + str(x[2]) + ", M=" + str(x[3]))
    _df = split_preform_answers(_df)
    n = _df.shape[0]
    print("_df shape:", _df.shape)

    # ANTES DO SPLIT (ESSENCIAL)
    counts = _df.groupby("PlayerId").size()
    valid_users = counts[counts >= 2].index
    _df = _df[_df["PlayerId"].isin(valid_users)]

    scaler__ = MinMaxScaler()

    # SPLIT
    train_df, test_df = leave_one_out_split(_df)

    # Normalizar
    numeric_cols = train_df.select_dtypes(include=["float64", "int64"]).columns
    cols_to_exclude = ["PlayerID", "PreFormAnswers", "LevelName"]
    numeric_cols = [c for c in numeric_cols if c not in cols_to_exclude]
    #scaler = MinMaxScaler()
    #train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
    #test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])v
    # Dicionário para guardar scalers por coluna
    scalers = {}

    for col in numeric_cols:
        scaler = MinMaxScaler()
        # Reshape para 2D
        values = train_df[col].values.reshape(-1, 1)
        scaler.fit(values)
        scalers[col] = scaler
        # Aplica transformação
        train_df[col] = scaler.transform(values).flatten()
        test_df[col] = scaler.transform(test_df[col].values.reshape(-1, 1)).flatten()


    # Augmentation
    _dfnoise = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfscale = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfnoise2 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 2*n)
    _dfscale2 = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfscale2 = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)
    _dfmixed = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfmixed = pd.concat([_dfmixed, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 6*n)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.10)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.9)], ignore_index=True)

    dataframes = [#_df, _dfnoise, _dfscale, _dfnoise2, _dfscale2, _dfmixed,
                  _dfmixed10]
    experiment_names = [#'Baseline', '1xN', '1xS-1_05', '2xN', '2xS-1_05+0_95', '1xS-1_05+1xN',
                        '6xN+4xS-1_05+0_95+1_1+0_9']

    strategies = [#'Original', '1xNoise', '1xScale', '2xN', '2xScale', '1xN+1xS',
                  '6xN+4xS']

    # Quantos níveis diferentes cada usuário jogou
    levels_per_user = _df.groupby("PlayerId")["LevelName"].nunique()

    # Filtrar apenas quem jogou mais de 1 nível
    multi_level_users = levels_per_user[levels_per_user > 1]

    print("Usuários que jogaram mais de um nível:")
    print(multi_level_users)

    # Filtragem colaborativa
    collaborative_filtering(_dfmixed10, method, test_df, recomenda, scalers)


def main(method, recomenda):

    initialize_data_frame()
    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()

    if(method == 0):
      inspect_dataset(user_frame, dungeon_frame)
    else:
      dungeon_frame = pd.merge(dungeon_frame, user_frame[['PlayerId', 'PreFormAnswers']], on=['PlayerId'], how='left')
      dungeon_frame = drop_invalid_rows(dungeon_frame)
      dungeon_frame = get_percentages(dungeon_frame)
      run(dungeon_frame, method, recomenda)


In [213]:
def aggregate_recommendations_generic2(
    recommended_levels_df,
    dungeon_metadata_df,
    user_profile_df,
    ratings_df,
    columns_to_use,
    metric_function,
    decision_function,
    top_n=5,
    scaler__=None
):
    """
    Versão 2 de aggregate_recommendations_generic que suporta o scaler para desnormalização
    """
    available_cols = [col for col in columns_to_use if col in dungeon_metadata_df.columns]

    if len(available_cols) == 0:
        print("Nenhuma coluna encontrada")
        return pd.DataFrame()

    resultado = []

    for user in recommended_levels_df['User'].unique():

        # Top N recomendações
        user_recs = (
            recommended_levels_df[recommended_levels_df['User'] == user]
            .head(top_n)
            .copy()
        )

        if len(user_recs) == 0:
            continue

        # Merge metadata
        user_recs = user_recs.merge(
            dungeon_metadata_df[['LevelName'] + available_cols],
            left_on='Item',
            right_on='LevelName',
            how='left'
        )

        # Merge rating
        user_recs = user_recs.merge(
            ratings_df[ratings_df['User'] == user][['Item', 'Rating']],
            on='Item',
            how='left'
        )

        rating_mean = user_recs['Rating'].mean()

        # Perfil
        user_profile = user_profile_df[user_profile_df['PlayerId'] == user]
        preform = user_profile['PreFormAnswers'].iloc[0] if len(user_profile) > 0 else 'N/A'

        # Métricas específicas - PASSA O SCALER
        metrics = metric_function(user_recs, scaler__)

        # Decisão específica
        recomendacao = decision_function(rating_mean, metrics)

        resultado.append({
            'User': user,
            'PreFormAnswers': preform,
            'rating_medio': round(rating_mean, 2),
            **metrics,
            'recomendacao': recomendacao
        })

    return pd.DataFrame(resultado)

def run_aggregation2(
    title,
    aggregation_function,
    rec_df,
    _df,
    user_profiles,
    collab_df,
    columns_to_print,
    extra_param=None,
    top_n=5,
    scaler__=None
):
    """
    Versão 2 de run_aggregation que suporta o scaler para desnormalização
    """
    if extra_param is not None:
        result = aggregation_function(
            rec_df,
            extra_param,
            user_profiles,
            collab_df,
            top_n=top_n,
            scaler__=scaler__
        )
    else:
        result = aggregation_function(
            rec_df,
            _df,
            user_profiles,
            collab_df,
            top_n=top_n,
            scaler__=scaler__
        )

    print(f"\n=== {title} ===")

    if len(result) > 0:
        print(result[columns_to_print].to_string())

    return result

def enemy_metrics2(user_recs, scaler):
    """
    Retorna a MÉDIA REAL de inimigos (desnormalizada)
    """
    if 'TotalEnemies' in user_recs.columns:
        if scaler is not None:
            # Pega os valores normalizados
            norm_values = user_recs['TotalEnemies'].values.reshape(-1, 1)
            # Desnormaliza
            real_values = scaler.inverse_transform(norm_values)
            return {
                'total_enemies_medio': real_values.mean()
            }
        else:
            return {
                'total_enemies_medio': user_recs['TotalEnemies'].mean()
            }
    return {'total_enemies_medio': 0}

def npc_metrics2(user_recs, scaler):
    """
    Retorna a MÉDIA REAL de NPCs (desnormalizada)
    """
    if 'TotalNpcs' in user_recs.columns:
        if scaler is not None:
            norm_values = user_recs['TotalNpcs'].values.reshape(-1, 1)
            real_values = scaler.inverse_transform(norm_values)
            return {
                'total_npcs_medio': real_values.mean()
            }
        else:
            return {
                'total_npcs_medio': user_recs['TotalNpcs'].mean()
            }
    return {'total_npcs_medio': 0}

def progress_metrics2(user_recs, scaler):
    """
    Retorna a MÉDIA REAL de tesouros (desnormalizada)
    """
    if 'TotalTreasure' in user_recs.columns:
        if scaler is not None:
            norm_values = user_recs['TotalTreasure'].values.reshape(-1, 1)
            real_values = scaler.inverse_transform(norm_values)
            return {
                'total_treasure_medio': real_values.mean()
            }
        else:
            return {
                'total_treasure_medio': user_recs['TotalTreasure'].mean()
            }
    return {'total_treasure_medio': 0}

def structure_metrics2(user_recs, scaler__):
    """
    Retorna a MÉDIA REAL de salas (desnormalizada)
    """
    if 'TotalRooms' in user_recs.columns:
        if scaler__ is not None:
            norm_values = user_recs['TotalRooms'].values.reshape(-1, 1)
            real_values = scaler__.inverse_transform(norm_values)
            return {
                'total_rooms_medio': real_values.mean()
            }
        else:
            return {
                'total_rooms_medio': user_recs['TotalRooms'].mean()
            }
    return {'total_rooms_medio': 0}

def enemy_decision2(rating_mean, metrics):
    total_enemies = metrics['total_enemies_medio']
    enemies_int = int(round(total_enemies))

    if rating_mean >= 0.7:
        if enemies_int >= 30:
            return f"MUITOS inimigos ({enemies_int})"
        elif enemies_int >= 15:
            return f"MEDIA quantidade ({enemies_int})"
        else:
            return f"POUCOS inimigos ({enemies_int})"
    else:
        return f"MUITO POUCOS inimigos ({enemies_int})"

def npc_decision2(rating_mean, metrics):
    total_npcs = metrics['total_npcs_medio']
    npcs_int = int(round(total_npcs))

    if rating_mean >= 0.7:
        if npcs_int >= 20:
            return f"MUITOS NPCs ({npcs_int})"
        elif npcs_int >= 10:
            return f"MEDIA quantidade ({npcs_int})"
        else:
            return f"POUCOS NPCs ({npcs_int})"
    else:
        return f"MUITO POUCOS NPCs ({npcs_int})"

def progress_decision2(rating_mean, metrics):
    total_treasure = metrics['total_treasure_medio']
    treasure_int = int(round(total_treasure))

    if rating_mean >= 0.7:
        if treasure_int >= 50:
            return f"MUITOS tesouros ({treasure_int})"
        elif treasure_int >= 25:
            return f"MEDIA quantidade de tesouros ({treasure_int})"
        else:
            return f"POUCOS tesouros ({treasure_int})"
    else:
        return f"POUCOS tesouros ({treasure_int})"

def structure_decision2(rating_mean, metrics):
    total_rooms = metrics['total_rooms_medio']
    rooms_int = int(round(total_rooms))

    if rating_mean >= 0.7:
        if rooms_int >= 20:
            return f"DUNGEONS GRANDES ({rooms_int} salas)"
        elif rooms_int >= 10:
            return f"DUNGEONS MEDIAS ({rooms_int} salas)"
        else:
            return f"DUNGEONS PEQUENAS ({rooms_int} salas)"
    else:
        return f"DUNGEONS PEQUENAS ({rooms_int} salas)"

def recomenda_tabela2(_df, collab_df, rec_user_df, scaler__):
    """
    Versão 2 de recomenda_tabela usando as funções com desnormalização
    """
    quest_columns = [
        'CompletedCreativityQuests',
        'CompletedKillQuests',
        'CompletedExploreQuests',
        'CompletedMasteryQuests',
        'CompletedImmersionQuests',
        'CompletedReportQuests',
        'CompletedListenQuests',
        'CompletedExchangeQuests',
        'CompletedGiveQuests',
        'CompletedGoToQuests',
        'CompletedGatherQuests'
    ]

    available_quests = [col for col in quest_columns if col in _df.columns]
    dungeon_quests = _df[['LevelName'] + available_quests].drop_duplicates()
    user_profiles = _df[['PlayerId', 'PreFormAnswers']].drop_duplicates()

    # Agregação de QUESTS (usa a original)
    agregacao_quests = run_aggregation2("RECOMENDACOES AGREGADAS POR PREFERENCIA DE QUESTS",
        lambda rec_df, meta_df, user_profiles, collab_df, top_n, scaler__=None:
         aggregate_recommendations_generic2(
             rec_df,
             meta_df,
             user_profiles,
             collab_df,
             columns_to_use=quest_columns,
             metric_function=quest_metrics,
             decision_function=quest_decision,
             top_n=top_n,
             scaler__=scaler__
         ), rec_user_df, _df, user_profiles,
       collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
       extra_param=dungeon_quests,
       top_n=5,
       scaler__=scaler__
    )

    # Agregação de INIMIGOS (usa as novas funções)
    agregacao_inimigos = run_aggregation2("RECOMENDACOES AGREGADAS POR DIFICULDADE DE INIMIGOS",
        lambda rec_df, meta_df, user_profiles, collab_df, top_n, scaler__=None:
         aggregate_recommendations_generic2(
             rec_df,
             meta_df,
             user_profiles,
             collab_df,
             columns_to_use=['TotalEnemies'],
             metric_function=enemy_metrics2,
             decision_function=enemy_decision2,
             top_n=top_n,
             scaler__=scaler__
         ), rec_user_df, _df, user_profiles,
       collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
       top_n=5,
       scaler__=scaler__
    )

    # Agregação de NPCS (usa as novas funções)
    agregacao_npcs = run_aggregation2("RECOMENDACOES AGREGADAS POR NPCS",
        lambda rec_df, meta_df, user_profiles, collab_df, top_n, scaler__=None:
         aggregate_recommendations_generic2(
             rec_df,
             meta_df,
             user_profiles,
             collab_df,
             columns_to_use=['TotalNpcs'],
             metric_function=npc_metrics2,
             decision_function=npc_decision2,
             top_n=top_n,
             scaler__=scaler__
         ), rec_user_df, _df, user_profiles,
       collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
       top_n=5,
       scaler__=scaler__
    )

    # Agregação de PROGRESSO (usa as novas funções)
    agregacao_progresso = run_aggregation2("RECOMENDACOES AGREGADAS POR PROGRESSO (TESOUROS/TEMPO)",
        lambda rec_df, meta_df, user_profiles, collab_df, top_n, scaler__=None:
         aggregate_recommendations_generic2(
             rec_df,
             meta_df,
             user_profiles,
             collab_df,
             columns_to_use=['TotalTreasure'],
             metric_function=progress_metrics2,
             decision_function=progress_decision2,
             top_n=top_n,
             scaler__=scaler__
         ), rec_user_df, _df, user_profiles,
       collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
       top_n=5,
       scaler__=scaler__
    )

    # Agregação de ESTRUTURA (usa as novas funções)
    agregacao_estrutura = run_aggregation2("RECOMENDACOES AGREGADAS POR ESTRUTURA (SALAS)",
        lambda rec_df, meta_df, user_profiles, collab_df, top_n, scaler__=None:
         aggregate_recommendations_generic2(
             rec_df,
             meta_df,
             user_profiles,
             collab_df,
             columns_to_use=['TotalRooms'],
             metric_function=structure_metrics2,
             decision_function=structure_decision2,
             top_n=top_n,
             scaler__=scaler__
         ), rec_user_df, _df, user_profiles,
       collab_df, ['User', 'PreFormAnswers', 'rating_medio', 'recomendacao'],
       top_n=5,
       scaler__=scaler__
    )

    # DataFrame com os VALORES REAIS (desnormalizados)
    df_valores_reais = pd.DataFrame({
        'User': agregacao_quests['User'],
        'PreFormAnswers': agregacao_quests['PreFormAnswers'],
        'rating_medio': agregacao_quests['rating_medio'],

        # VALORES REAIS DESNORMALIZADOS
        'media_inimigos': agregacao_inimigos['total_enemies_medio'],
        'media_npcs': agregacao_npcs['total_npcs_medio'],
        'media_tesouros': agregacao_progresso['total_treasure_medio'],
        'media_salas': agregacao_estrutura['total_rooms_medio'],

        # Decisões com valores reais
        'decisao_inimigos': agregacao_inimigos['recomendacao'],
        'decisao_npcs': agregacao_npcs['recomendacao'],
        'decisao_progresso': agregacao_progresso['recomendacao'],
        'decisao_estrutura': agregacao_estrutura['recomendacao'],

        # Dados das quests
        'quest_primaria': agregacao_quests['quest_primaria'],
        'score_primario': agregacao_quests['score_primario'],
    })

    # Exibir os valores reais
    print("\n=== VALORES REAIS DAS MÉTRICAS (DESNORMALIZADOS) ===")
    print(df_valores_reais[['User', 'media_inimigos', 'media_npcs', 'media_tesouros', 'media_salas']])

    print("\n=== DECISÕES COM VALORES REAIS ===")
    print(df_valores_reais[['User', 'decisao_inimigos', 'decisao_npcs', 'decisao_progresso', 'decisao_estrutura']])

    return df_valores_reais

In [214]:
def combine_recommendations(
    quests_df,
    enemies_df,
    npcs_df,
    structure_df,
    dungeon_metadata_df,
    user_id=None,
    top_n=5
):
    """
    Combina as recomendações de múltiplas estratégias para gerar:
    1. Recomendação de nível existente (mais adequado)
    2. Sugestão de configuração (nova dungeon)
    """

    # 1. Pegar a recomendação principal de cada estratégia
    combined_scores = {}

    # Dicionário para armazenar as configurações sugeridas
    suggested_config = {
        'quest_type': None,
        'enemy_density': None,
        'npc_density': None,
        'structure_size': None,
        'key_range': None
    }

    # --- Quest Recommendation ---
    if quests_df is not None and len(quests_df) > 0:
        top_quest = quests_df.iloc[0]
        suggested_config['quest_type'] = top_quest['recomendacao']
        # Para níveis existentes, extrair os top levels que atendem essa quest
        quest_levels = get_levels_by_quest_preference(
            dungeon_metadata_df,
            top_quest['recomendacao']
        )
        for level in quest_levels:
            combined_scores[level] = combined_scores.get(level, 0) + 0.25

    # --- Enemy Recommendation ---
    if enemies_df is not None and len(enemies_df) > 0:
        top_enemy = enemies_df.iloc[0]
        suggested_config['enemy_density'] = top_enemy['recomendacao']
        enemy_levels = get_levels_by_enemy_density(
            dungeon_metadata_df,
            top_enemy['recomendacao']
        )
        for level in enemy_levels:
            combined_scores[level] = combined_scores.get(level, 0) + 0.25

    # --- NPC Recommendation ---
    if npcs_df is not None and len(npcs_df) > 0:
        top_npc = npcs_df.iloc[0]
        suggested_config['npc_density'] = top_npc['recomendacao']
        npc_levels = get_levels_by_npc_density(
            dungeon_metadata_df,
            top_npc['recomendacao']
        )
        for level in npc_levels:
            combined_scores[level] = combined_scores.get(level, 0) + 0.25

    # --- Structure Recommendation ---
    if structure_df is not None and len(structure_df) > 0:
        top_structure = structure_df.iloc[0]
        suggested_config['structure_size'] = top_structure['recomendacao']
        suggested_config['key_range'] = get_key_range_from_structure(top_structure)
        structure_levels = get_levels_by_structure(
            dungeon_metadata_df,
            top_structure['recomendacao']
        )
        for level in structure_levels:
            combined_scores[level] = combined_scores.get(level, 0) + 0.25

    # 2. Ordenar níveis existentes por score combinado
    ranked_levels = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)

    # 3. Formatar resultado
    result = {
        'user_id': user_id,
        'existing_level_recommendations': [],
        'suggested_configuration': suggested_config,
        'confidence_scores': {}
    }

    for level, score in ranked_levels[:top_n]:
        # Buscar metadata completa do level
        level_metadata = dungeon_metadata_df[
            dungeon_metadata_df['LevelName'] == level
        ].iloc[0] if len(dungeon_metadata_df[dungeon_metadata_df['LevelName'] == level]) > 0 else {}

        result['existing_level_recommendations'].append({
            'level_name': level,
            'combined_score': round(score, 3),
            'metadata': {
                'TotalRooms': level_metadata.get('TotalRooms', 'N/A'),
                'TotalKeys': level_metadata.get('TotalKeys', 'N/A'),
                'TotalEnemies': level_metadata.get('TotalEnemies', 'N/A'),
                'TotalNpcs': level_metadata.get('TotalNpcs', 'N/A')
            }
        })

        result['confidence_scores'][level] = round(score, 3)

    return result


def get_levels_by_quest_preference(dungeon_df, quest_preference):
    """Retorna níveis que correspondem à preferência de quests"""
    # Mapeamento de preferência -> colunas de quest
    quest_mapping = {
        'combate': 'CompletedKillQuests',
        'exploracao': 'CompletedExploreQuests',
        'criatividade': 'CompletedCreativityQuests',
        'maestria': 'CompletedMasteryQuests',
        'imersao': 'CompletedImmersionQuests'
    }

    if quest_preference in quest_mapping:
        col = quest_mapping[quest_preference]
        if col in dungeon_df.columns:
            # Pega os níveis com maior média na quest preferida
            top_levels = dungeon_df.groupby('LevelName')[col].mean().sort_values(ascending=False).head(10)
            return top_levels.index.tolist()

    return dungeon_df['LevelName'].unique().tolist()[:10]


def get_levels_by_enemy_density(dungeon_df, enemy_preference):
    """Retorna níveis baseado na densidade de inimigos"""
    # enemy_preference pode ser: "MUITOS inimigos", "MEDIA quantidade", "POUCOS inimigos"
    density_map = {
        'MUITOS inimigos': (0.7, 1.0),
        'MEDIA quantidade': (0.4, 0.7),
        'POUCOS inimigos': (0.2, 0.4),
        'MUITO POUCOS inimigos': (0.0, 0.2)
    }

    if 'EnemyKillRate' in dungeon_df.columns:
        low, high = density_map.get(enemy_preference, (0, 1))
        filtered = dungeon_df[
            (dungeon_df['EnemyKillRate'] >= low) &
            (dungeon_df['EnemyKillRate'] < high)
        ]
        return filtered['LevelName'].unique().tolist()[:10]

    return dungeon_df['LevelName'].unique().tolist()[:10]


def get_levels_by_npc_density(dungeon_df, npc_preference):
    """Retorna níveis baseado na densidade de NPCs"""
    density_map = {
        'MUITOS NPCs': (0.7, 1.0),
        'MEDIA quantidade': (0.4, 0.7),
        'POUCOS NPCs': (0.2, 0.4),
        'MUITO POUCOS NPCs': (0.0, 0.2)
    }

    if 'NpcsInteracted' in dungeon_df.columns and 'TotalNpcs' in dungeon_df.columns:
        dungeon_df['NpcRatio'] = dungeon_df['NpcsInteracted'] / (dungeon_df['TotalNpcs'] + 1)
        low, high = density_map.get(npc_preference, (0, 1))
        filtered = dungeon_df[
            (dungeon_df['NpcRatio'] >= low) &
            (dungeon_df['NpcRatio'] < high)
        ]
        return filtered['LevelName'].unique().tolist()[:10]

    return dungeon_df['LevelName'].unique().tolist()[:10]


def get_levels_by_structure(dungeon_df, structure_preference):
    """Retorna níveis baseado no tamanho da dungeon"""
    size_map = {
        'DUNGEONS GRANDES': (20, float('inf')),
        'DUNGEONS MEDIAS': (10, 20),
        'DUNGEONS PEQUENAS': (0, 10)
    }

    if 'TotalRooms' in dungeon_df.columns:
        low, high = size_map.get(structure_preference, (0, float('inf')))
        filtered = dungeon_df[
            (dungeon_df['TotalRooms'] >= low) &
            (dungeon_df['TotalRooms'] < high)
        ]
        return filtered['LevelName'].unique().tolist()[:10]

    return dungeon_df['LevelName'].unique().tolist()[:10]


def get_key_range_from_structure(structure_row):
    """Extrai faixa de chaves da recomendação de estrutura"""
    # structure_row tem 'keys_medio', 'min_chaves', 'max_chaves'
    if 'keys_medio' in structure_row:
        media = structure_row['keys_medio']
        if media <= 3:
            return "0-3 chaves"
        elif media <= 6:
            return "4-6 chaves"
        elif media <= 9:
            return "7-9 chaves"
        else:
            return "10+ chaves"
    return "N/A"


def generate_new_dungeon_config(suggested_config):
    """
    Gera uma configuração completa para uma nova dungeon
    baseada nas sugestões combinadas
    """
    config = {
        'dungeon_name': 'Suggested_Dungeon',
        'structural_attributes': {},
        'gameplay_attributes': {},
        'estimated_difficulty': 'MEDIUM'
    }

    # Configuração estrutural
    if suggested_config['structure_size'] == 'DUNGEONS GRANDES':
        config['structural_attributes']['TotalRooms'] = random.randint(20, 35)
        config['structural_attributes']['TotalKeys'] = random.randint(8, 15)
        config['estimated_difficulty'] = 'HARD'
    elif suggested_config['structure_size'] == 'DUNGEONS MEDIAS':
        config['structural_attributes']['TotalRooms'] = random.randint(10, 19)
        config['structural_attributes']['TotalKeys'] = random.randint(4, 9)
        config['estimated_difficulty'] = 'MEDIUM'
    else:
        config['structural_attributes']['TotalRooms'] = random.randint(3, 9)
        config['structural_attributes']['TotalKeys'] = random.randint(1, 4)
        config['estimated_difficulty'] = 'EASY'

    # Configuração de inimigos
    enemy_map = {
        'MUITOS inimigos': (0.7, 1.0, 'HIGH'),
        'MEDIA quantidade': (0.4, 0.7, 'MEDIUM'),
        'POUCOS inimigos': (0.2, 0.4, 'LOW'),
        'MUITO POUCOS inimigos': (0.0, 0.2, 'VERY_LOW')
    }

    if suggested_config['enemy_density'] in enemy_map:
        low, high, difficulty = enemy_map[suggested_config['enemy_density']]
        # TotalEnemies baseado no número de salas
        rooms = config['structural_attributes']['TotalRooms']
        enemies_per_room = random.uniform(low, high)
        config['gameplay_attributes']['TotalEnemies'] = int(rooms * enemies_per_room)
        config['gameplay_attributes']['EnemyKillRate_target'] = round(random.uniform(low, high), 2)

    # Configuração de NPCs
    npc_map = {
        'MUITOS NPCs': (0.7, 1.0),
        'MEDIA quantidade': (0.4, 0.7),
        'POUCOS NPCs': (0.2, 0.4),
        'MUITO POUCOS NPCs': (0.0, 0.2)
    }

    if suggested_config['npc_density'] in npc_map:
        low, high = npc_map[suggested_config['npc_density']]
        rooms = config['structural_attributes']['TotalRooms']
        npcs_per_room = random.uniform(low, high)
        config['gameplay_attributes']['TotalNpcs'] = int(rooms * npcs_per_room)

    # Quest type influence
    quest_to_attribute = {
        'combate': 'combat_focus',
        'exploracao': 'exploration_focus',
        'criatividade': 'puzzle_focus'
    }
    config['quest_focus'] = suggested_config.get('quest_type', 'balanced')

    return config


def run_combined_recommendation(
    _df,
    collab_df,
    rec_user_df,
    dungeon_metadata_df,
    user_id=None,
    top_n=5
):
    """
    Executa todas as agregações e combina os resultados
    """
    # 1. Executar todas as agregações
    quest_result = aggregate_recommendations_generic(
        rec_user_df,
        dungeon_metadata_df,
        _df[['PlayerId', 'PreFormAnswers']].drop_duplicates(),
        collab_df,
        columns_to_use=[
            'CompletedCreativityQuests', 'CompletedKillQuests',
            'CompletedExploreQuests', 'CompletedMasteryQuests',
            'CompletedImmersionQuests'
        ],
        metric_function=quest_metrics,
        decision_function=quest_decision,
        top_n=top_n

    )

    enemy_result = aggregate_recommendations_generic(
        rec_user_df,
        dungeon_metadata_df,
        _df[['PlayerId', 'PreFormAnswers']].drop_duplicates(),
        collab_df,
        columns_to_use=['EnemyKillRate'],
        metric_function=enemy_metrics,
        decision_function=enemy_decision,
        top_n=top_n
    )

    npc_result = aggregate_recommendations_generic(
        rec_user_df,
        dungeon_metadata_df,
        _df[['PlayerId', 'PreFormAnswers']].drop_duplicates(),
        collab_df,
        columns_to_use=['TotalNpcs', 'NpcsInteracted'],
        metric_function=npc_metrics,
        decision_function=npc_decision,
        top_n=top_n
    )

    structure_result = aggregate_recommendations_generic(
        rec_user_df,
        dungeon_metadata_df,
        _df[['PlayerId', 'PreFormAnswers']].drop_duplicates(),
        collab_df,
        columns_to_use=['TotalRooms', 'TotalKeys', 'TotalLocks', 'UniqueRoomsEntered', 'RoomsEntered'],
        metric_function=structure_metrics,
        decision_function=structure_decision,
        top_n=top_n
    )

    # 2. Combinar recomendações para o primeiro usuário (ou específico)
    if user_id is None and len(quest_result) > 0:
        user_id = quest_result.iloc[0]['User']

    # Filtrar para o usuário específico
    quest_user = quest_result[quest_result['User'] == user_id] if len(quest_result) > 0 else None
    enemy_user = enemy_result[enemy_result['User'] == user_id] if len(enemy_result) > 0 else None
    npc_user = npc_result[npc_result['User'] == user_id] if len(npc_result) > 0 else None
    structure_user = structure_result[structure_result['User'] == user_id] if len(structure_result) > 0 else None

    # 3. Combinar
    combined = combine_recommendations(
        quest_user,
        enemy_user,
        npc_user,
        structure_user,
        dungeon_metadata_df,
        user_id=user_id,
        top_n=top_n
    )

    # 4. Gerar sugestão de nova dungeon
    new_dungeon_config = generate_new_dungeon_config(combined['suggested_configuration'])

    # 5. Exibir resultados
    print(f"\n{'='*60}")
    print(f"RECOMENDAÇÃO COMBINADA PARA USUÁRIO {user_id}")
    print(f"{'='*60}")

    print("\n--- Perfil do Usuário ---")
    user_profile = _df[_df['PlayerId'] == user_id]['PreFormAnswers'].iloc[0] if len(_df[_df['PlayerId'] == user_id]) > 0 else 'N/A'
    print(f"PreFormAnswers: {user_profile}")

    print("\n--- Recomendações de Níveis Existentes ---")
    for rec in combined['existing_level_recommendations']:
        print(f"\n  📍 {rec['level_name']} (score: {rec['combined_score']})")
        print(f"     Salas: {rec['metadata']['TotalRooms']} | Chaves: {rec['metadata']['TotalKeys']}")
        print(f"     Inimigos: {rec['metadata']['TotalEnemies']} | NPCs: {rec['metadata']['TotalNpcs']}")

    print("\n--- Sugestão de Configuração para Nova Dungeon ---")
    print(f"  🏰 Tamanho: {combined['suggested_configuration']['structure_size']}")
    print(f"  ⚔️  Inimigos: {combined['suggested_configuration']['enemy_density']}")
    print(f"  💬 NPCs: {combined['suggested_configuration']['npc_density']}")
    print(f"  🔑 Chaves: {combined['suggested_configuration']['key_range']}")
    print(f"  🎯 Foco de Quest: {combined['suggested_configuration']['quest_type']}")

    print("\n--- Nova Dungeon Sugerida (Configuração Detalhada) ---")
    print(f"  Nome sugerido: {new_dungeon_config['dungeon_name']}")
    print(f"  Dificuldade estimada: {new_dungeon_config['estimated_difficulty']}")
    print(f"  Estrutura: {new_dungeon_config['structural_attributes']}")
    print(f"  Gameplay: {new_dungeon_config['gameplay_attributes']}")
    print(f"  Foco de Quest: {new_dungeon_config['quest_focus']}")

    return combined, new_dungeon_config

In [215]:
main(0, False)

Combinações únicas:
[(1, 3, 2, 4), (2, 1, 3, 4), (2, 1, 4, 3), (3, 2, 1, 4), (3, 2, 4, 1), (4, 1, 3, 2), (4, 2, 1, 3), (4, 3, 2, 1)]

Quantidade de combinações únicas:
8

Frequência de cada combinação:
PreFormAnswers
(4, 2, 1, 3)    3
(3, 2, 4, 1)    3
(1, 3, 2, 4)    3
(3, 2, 1, 4)    2
(4, 3, 2, 1)    1
(4, 1, 3, 2)    1
(2, 1, 3, 4)    1
(2, 1, 4, 3)    1
Name: count, dtype: int64

Frequência no dungeon_frame final:
PreFormAnswers
[1, 3, 2, 4]    26
[4, 1, 3, 2]    25
[2, 1, 3, 4]    24
[4, 2, 1, 3]    18
[3, 2, 1, 4]    11
[3, 2, 4, 1]     9
[2, 1, 4, 3]     5
[4, 3, 2, 1]     1
Name: count, dtype: int64

Total de colunas: 57

  1. CompletedAchievementQuests
  2. CompletedCreativityQuests
  3. CompletedDamageQuests
  4. CompletedExchangeQuests
  5. CompletedExploreQuests
  6. CompletedGatherQuests
  7. CompletedGiveQuests
  8. CompletedGoToQuests
  9. CompletedImmersionQuests
 10. CompletedKillQuests
 11. CompletedListenQuests
 12. CompletedMasteryQuests
 13. CompletedQuests
 14. C

In [216]:
def enemy_metrics2(user_recs, scaler):
    # Pega os valores reais de TotalEnemies do DataFrame original
    # (assumindo que o DataFrame tem os valores originais)
    if 'TotalEnemies' in user_recs.columns:
        # Se os dados estão normalizados, pega do scaler
        if scaler is not None:
            # Cria um array com os valores normalizados
            norm_values = user_recs['TotalEnemies'].values.reshape(-1, 1)
            # Desnormaliza
            real_values = scaler.inverse_transform(norm_values)
            return {
                'total_enemies_medio': real_values.mean()
            }
        else:
            return {
                'total_enemies_medio': user_recs['TotalEnemies'].mean()
            }
    return {'total_enemies_medio': 0}

def npc_metrics2(user_recs, scaler):
    if 'TotalNpcs' in user_recs.columns:
        if scaler is not None:
            norm_values = user_recs['TotalNpcs'].values.reshape(-1, 1)
            real_values = scaler.inverse_transform(norm_values)
            return {
                'total_npcs_medio': real_values.mean()
            }
        else:
            return {
                'total_npcs_medio': user_recs['TotalNpcs'].mean()
            }
    return {'total_npcs_medio': 0}

def progress_metrics2(user_recs, scaler):
    if 'TotalTreasure' in user_recs.columns:
        if scaler is not None:
            norm_values = user_recs['TotalTreasure'].values.reshape(-1, 1)
            real_values = scaler.inverse_transform(norm_values)
            return {
                'total_treasure_medio': real_values.mean()
            }
        else:
            return {
                'total_treasure_medio': user_recs['TotalTreasure'].mean()
            }
    return {'total_treasure_medio': 0}

def structure_metrics2(user_recs, scaler__):
    if 'TotalRooms' in user_recs.columns:
        if scaler__ is not None:
            norm_values = user_recs['TotalRooms'].values.reshape(-1, 1)
            real_values = scaler__.inverse_transform(norm_values)
            return {
                'total_rooms_medio': real_values.mean()
            }
        else:
            return {
                'total_rooms_medio': user_recs['TotalRooms'].mean()
            }
    return {'total_rooms_medio': 0}

In [217]:
def enemy_metrics2(user_recs, scaler_dict):
    """
    Retorna a MÉDIA REAL de inimigos (desnormalizada)
    """
    if 'TotalEnemies' in user_recs.columns:
        if scaler_dict is not None and 'TotalEnemies' in scaler_dict:
            # Pega os valores normalizados
            norm_values = user_recs['TotalEnemies'].values.reshape(-1, 1)
            # Desnormaliza usando o scaler específico da coluna
            real_values = scaler_dict['TotalEnemies'].inverse_transform(norm_values)
            return {
                'total_enemies_medio': real_values.mean()
            }
        else:
            return {
                'total_enemies_medio': user_recs['TotalEnemies'].mean()
            }
    return {'total_enemies_medio': 0}

def npc_metrics2(user_recs, scaler_dict):
    if 'TotalNpcs' in user_recs.columns:
        if scaler_dict is not None and 'TotalNpcs' in scaler_dict:
            norm_values = user_recs['TotalNpcs'].values.reshape(-1, 1)
            real_values = scaler_dict['TotalNpcs'].inverse_transform(norm_values)
            return {
                'total_npcs_medio': real_values.mean()
            }
        else:
            return {
                'total_npcs_medio': user_recs['TotalNpcs'].mean()
            }
    return {'total_npcs_medio': 0}

def progress_metrics2(user_recs, scaler_dict):
    if 'TotalTreasure' in user_recs.columns:
        if scaler_dict is not None and 'TotalTreasure' in scaler_dict:
            norm_values = user_recs['TotalTreasure'].values.reshape(-1, 1)
            real_values = scaler_dict['TotalTreasure'].inverse_transform(norm_values)
            return {
                'total_treasure_medio': real_values.mean()
            }
        else:
            return {
                'total_treasure_medio': user_recs['TotalTreasure'].mean()
            }
    return {'total_treasure_medio': 0}

def structure_metrics2(user_recs, scaler_dict):
    if 'TotalRooms' in user_recs.columns:
        if scaler_dict is not None and 'TotalRooms' in scaler_dict:
            norm_values = user_recs['TotalRooms'].values.reshape(-1, 1)
            real_values = scaler_dict['TotalRooms'].inverse_transform(norm_values)
            return {
                'total_rooms_medio': real_values.mean()
            }
        else:
            return {
                'total_rooms_medio': user_recs['TotalRooms'].mean()
            }
    return {'total_rooms_medio': 0}



In [218]:
def enemy_decision2(rating_mean, metrics):
    total_enemies = metrics['total_enemies_medio']
    enemies_int = int(round(total_enemies))  # Arredonda para inteiro

    if rating_mean >= 0.7:
        if enemies_int >= 30:
            return f"MUITOS inimigos ({enemies_int})"
        elif enemies_int >= 15:
            return f"MEDIA quantidade ({enemies_int})"
        else:
            return f"POUCOS inimigos ({enemies_int})"
    else:
        return f"MUITO POUCOS inimigos ({enemies_int})"

def npc_decision2(rating_mean, metrics):
    total_npcs = metrics['total_npcs_medio']
    npcs_int = int(round(total_npcs))

    if rating_mean >= 0.7:
        if npcs_int >= 20:
            return f"MUITOS NPCs ({npcs_int})"
        elif npcs_int >= 10:
            return f"MEDIA quantidade ({npcs_int})"
        else:
            return f"POUCOS NPCs ({npcs_int})"
    else:
        return f"MUITO POUCOS NPCs ({npcs_int})"

def progress_decision2(rating_mean, metrics):
    total_treasure = metrics['total_treasure_medio']
    treasure_int = int(round(total_treasure))

    if rating_mean >= 0.7:
        if treasure_int >= 50:
            return f"MUITOS tesouros ({treasure_int})"
        elif treasure_int >= 25:
            return f"MEDIA quantidade de tesouros ({treasure_int})"
        else:
            return f"POUCOS tesouros ({treasure_int})"
    else:
        return f"POUCOS tesouros ({treasure_int})"

def structure_decision2(rating_mean, metrics):
    total_rooms = metrics['total_rooms_medio']
    rooms_int = int(round(total_rooms))

    if rating_mean >= 0.7:
        if rooms_int >= 20:
            return f"DUNGEONS GRANDES ({rooms_int} salas)"
        elif rooms_int >= 10:
            return f"DUNGEONS MEDIAS ({rooms_int} salas)"
        else:
            return f"DUNGEONS PEQUENAS ({rooms_int} salas)"
    else:
        return f"DUNGEONS PEQUENAS ({rooms_int} salas)"

In [219]:
def npc_decision(rating_mean, metrics):
    """USA APENAS O VALOR REAL DE NPCS para classificar"""
    total_npcs = metrics['interacao_ratio']
    npcs_int = int(round(total_npcs))

    # Classifica SOMENTE pelo número real
    if npcs_int >= 20:
        return f"MUITOS NPCs ({npcs_int})"
    elif npcs_int >= 10:
        return f"MEDIA quantidade ({npcs_int})"
    elif npcs_int >= 5:
        return f"POUCOS NPCs ({npcs_int})"
    else:
        return f"MUITO POUCOS NPCs ({npcs_int})"

In [220]:
def npc_decision2(rating_mean, metrics):
    """USA RATING + VALOR REAL para decidir a recomendação final"""
    total_npcs = metrics['total_npcs_medio']
    npcs_int = int(round(total_npcs))

    # Se rating for ALTO (>= 0.7), classifica de forma mais positiva
    if rating_mean >= 0.7:
        if npcs_int >= 10:
            return f"MUITOS NPCs ({npcs_int}) - bem avaliado"
        elif npcs_int >= 5:
            return f"MEDIA quantidade ({npcs_int}) - bem avaliado"
        else:
            return f"POUCOS NPCs ({npcs_int}) - bem avaliado"
    # Se rating for BAIXO (< 0.7), classifica de forma mais negativa
    else:
        if npcs_int >= 20:
            return f"MEDIA quantidade ({npcs_int}) - mal avaliado"
        elif npcs_int >= 10:
            return f"POUCOS NPCs ({npcs_int}) - mal avaliado"
        else:
            return f"MUITO POUCOS NPCs ({npcs_int}) - mal avaliado"

In [229]:
def enemy_decision(rating_mean, metrics):
    total_enemies = metrics['TotalEnemies']
    enemies_int = int(round(total_enemies))

    if enemies_int >= 50:
        return f"MUITOS inimigos ({enemies_int})"
    elif enemies_int >= 30:
        return f"MEDIA quantidade ({enemies_int})"
    elif enemies_int >= 15:
        return f"POUCOS inimigos ({enemies_int})"
    else:
        return f"MUITO POUCOS inimigos ({enemies_int})"

def npc_decision(rating_mean, metrics):
    total_npcs = metrics['interacao_ratio']
    npcs_int = int(round(total_npcs))

    if npcs_int >= 20:
        return f"MUITOS NPCs ({npcs_int})"
    elif npcs_int >= 10:
        return f"MEDIA quantidade ({npcs_int})"
    elif npcs_int >= 5:
        return f"POUCOS NPCs ({npcs_int})"
    else:
        return f"MUITO POUCOS NPCs ({npcs_int})"

'''def progress_decision(rating_mean, metrics):
    total_treasure = metrics['treasure_ratio']
    treasure_int = int(round(total_treasure))

    if treasure_int >= 30:
        return f"MUITOS tesouros ({treasure_int})"
    elif treasure_int >= 20:
        return f"MEDIA quantidade de tesouros ({treasure_int})"
    elif treasure_int >= 10:
        return f"POUCOS tesouros ({treasure_int})"
    else:
        return f"MUITO POUCOS tesouros ({treasure_int})"'''
def progress_decision(rating_mean, metrics):
    total_treasure = metrics['treasure_ratio']

    # Verifica se é infinito ou NaN
    if np.isinf(total_treasure) or np.isnan(total_treasure):
        return "SEM TESOUROS DISPONÍVEIS"

    treasure_int = int(round(total_treasure))

    if treasure_int >= 30:
        return f"MUITOS tesouros ({treasure_int})"
    elif treasure_int >= 20:
        return f"MEDIA quantidade de tesouros ({treasure_int})"
    elif treasure_int >= 10:
        return f"POUCOS tesouros ({treasure_int})"
    else:
        return f"MUITO POUCOS tesouros ({treasure_int})"

def structure_decision(rating_mean, metrics):
    total_rooms = metrics['rooms_medio']
    rooms_int = int(round(total_rooms))

    if rooms_int >= 25:
        return f"DUNGEONS GRANDES ({rooms_int} salas)"
    elif rooms_int >= 15:
        return f"DUNGEONS MEDIAS ({rooms_int} salas)"
    else:
        return f"DUNGEONS PEQUENAS ({rooms_int} salas)"

In [225]:
def enemy_decision2(rating_mean, metrics):
    total_enemies = metrics['total_enemies_medio']
    enemies_int = int(round(total_enemies))

    if rating_mean >= 0.7:
        if enemies_int >= 50:
            return f"MUITOS inimigos ({enemies_int}) ⭐"
        elif enemies_int >= 30:
            return f"MEDIA quantidade ({enemies_int}) ⭐"
        elif enemies_int >= 15:
            return f"POUCOS inimigos ({enemies_int}) ⭐"
        else:
            return f"MUITO POUCOS inimigos ({enemies_int}) ⭐"
    else:
        if enemies_int >= 50:
            return f"MUITOS inimigos ({enemies_int}) 👎"
        elif enemies_int >= 30:
            return f"MEDIA quantidade ({enemies_int}) 👎"
        elif enemies_int >= 15:
            return f"POUCOS inimigos ({enemies_int}) 👎"
        else:
            return f"MUITO POUCOS inimigos ({enemies_int}) 👎"

def npc_decision2(rating_mean, metrics):
    total_npcs = metrics['total_npcs_medio']
    npcs_int = int(round(total_npcs))

    if rating_mean >= 0.7:
        if npcs_int >= 20:
            return f"MUITOS NPCs ({npcs_int}) ⭐"
        elif npcs_int >= 10:
            return f"MEDIA quantidade ({npcs_int}) ⭐"
        elif npcs_int >= 5:
            return f"POUCOS NPCs ({npcs_int}) ⭐"
        else:
            return f"MUITO POUCOS NPCs ({npcs_int}) ⭐"
    else:
        if npcs_int >= 20:
            return f"MUITOS NPCs ({npcs_int}) 👎"
        elif npcs_int >= 10:
            return f"MEDIA quantidade ({npcs_int}) 👎"
        elif npcs_int >= 5:
            return f"POUCOS NPCs ({npcs_int}) 👎"
        else:
            return f"MUITO POUCOS NPCs ({npcs_int}) 👎"

def progress_decision2(rating_mean, metrics):
    total_treasure = metrics['total_treasure_medio']
    treasure_int = int(round(total_treasure))

    if rating_mean >= 0.7:
        if treasure_int >= 30:
            return f"MUITOS tesouros ({treasure_int}) ⭐"
        elif treasure_int >= 20:
            return f"MEDIA quantidade ({treasure_int}) ⭐"
        elif treasure_int >= 10:
            return f"POUCOS tesouros ({treasure_int}) ⭐"
        else:
            return f"MUITO POUCOS tesouros ({treasure_int}) ⭐"
    else:
        if treasure_int >= 30:
            return f"MUITOS tesouros ({treasure_int}) 👎"
        elif treasure_int >= 20:
            return f"MEDIA quantidade ({treasure_int}) 👎"
        elif treasure_int >= 10:
            return f"POUCOS tesouros ({treasure_int}) 👎"
        else:
            return f"MUITO POUCOS tesouros ({treasure_int}) 👎"

def structure_decision2(rating_mean, metrics):
    total_rooms = metrics['total_rooms_medio']
    rooms_int = int(round(total_rooms))

    if rating_mean >= 0.7:
        if rooms_int >= 25:
            return f"DUNGEONS GRANDES ({rooms_int} salas) ⭐"
        elif rooms_int >= 15:
            return f"DUNGEONS MEDIAS ({rooms_int} salas) ⭐"
        else:
            return f"DUNGEONS PEQUENAS ({rooms_int} salas) ⭐"
    else:
        if rooms_int >= 25:
            return f"DUNGEONS GRANDES ({rooms_int} salas) 👎"
        elif rooms_int >= 15:
            return f"DUNGEONS MEDIAS ({rooms_int} salas) 👎"
        else:
            return f"DUNGEONS PEQUENAS ({rooms_int} salas) 👎"

In [230]:
main(21, False)

/tmp/ipykernel_4658/576070251.py:92: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _data_frame.replace([np.inf, -np.inf], 0, inplace=True)


_df shape: (119, 67)
Usuários que jogaram mais de um nível:
PlayerId
53627948       4
78194049       3
81972292      11
121412094     12
282268519      4
1093542367    12
1153731241     4
1776427228     2
2079201442     4
2079942857    12
2096844718     3
Name: LevelName, dtype: int64
-------------------- VISUALIZAÇÃO DOS DADOS ----------------------------
(1710, 67)
                      PostFormAnswers  TotalEnemies  TotalMasteryQuests  MaxCombo  TotalExploreQuests  TotalNpcs  CompletedReportQuests  CompletedListenQuests  TotalDeaths  CompletedAchievementQuests  TotalGiveQuests  TotalAchievementQuests  TotalKeys  PlayerId  TotalLocks  KeysCollected  TotalExchangeQuests  TotalLostHealth  CompletedExploreQuests  CompletedExchangeQuests  TotalGoToQuests  RoomsEntered  CompletedDamageQuests  TotalAttempts  TotalCreativityQuests  TotalGatherQuests  NpcsInteracted  CompletedMasteryQuests                                       InputProfile  CompletedGiveQuests WeaponName  TotalImmersionQuest

KeyError: 'TotalEnemies'



## Atributos Estruturais
1. **TotalRooms** - número de salas
2. **TotalKeys** - número de chaves
3. **TotalLocks** - número de fechaduras
4. **UniqueRoomsEntered** - salas únicas visitadas
5. **RoomsEntered** - total de salas entradas (com revisitas)

## Atributos de Combate
6. **TotalEnemies** - número total de inimigos
7. **EnemiesKilled** - inimigos mortos
8. **MaxCombo** - maior combo
9. **TotalLostHealth** - vida perdida
10. **TotalDeaths** - número de mortes

## Atributos de Interação
11. **TotalNpcs** - número de NPCs
12. **NpcsInteracted** - NPCs com quem interagiu

## Atributos de Progresso
13. **TotalTreasure** - tesouros disponíveis
14. **TreasuresCollected** - tesouros coletados
15. **TimeToFinish** - tempo para finalizar

## Atributos de Missões (Quests)
16. **TotalAchievementQuests** - total de quests de conquista
17. **TotalCreativityQuests** - total de quests de criatividade
18. **TotalDamageQuests** - total de quests de dano
19. **TotalExploreQuests** - total de quests de exploração
20. **TotalGatherQuests** - total de quests de coleta
21. **TotalGiveQuests** - total de quests de entrega
22. **TotalGoToQuests** - total de quests de "ir até"
23. **TotalImmersionQuests** - total de quests de imersão
24. **TotalKillQuests** - total de quests de matar
25. **TotalListenQuests** - total de quests de ouvir
26. **TotalMasteryQuests** - total de quests de mestria
27. **TotalReadQuests** - total de quests de ler
28. **TotalReportQuests** - total de quests de relatório

## Métricas Calculadas
29. **LockUsage** - uso de fechaduras (LocksOpened/TotalLocks)
30. **KeyCollection** - coleta de chaves (KeysCollected/TotalKeys)
31. **MapCompletion** - exploração (UniqueRoomsEntered/TotalRooms)
32. **RoomRevisitRate** - taxa de revisita
33. **EnemyKillRate** - taxa de abate (EnemiesKilled/TotalEnemies)
34. **Success Rate** - taxa de sucesso (Wins/Attempts)



#TODO
- 1º FAZER RECOMENDACAO MULTI

-- recomendar level e explica - recomendado pq tem x salsa, filtrar

-- criar item composto ex: D: 10, C: 20, I: 4 - DIFICUA, CHAVE, INIMIGO... - FAZRE UM TIPO DE NOME PARA CASOS - AI USAR COLUNA

-- ANTES DE IR RECOMENDADO PERIMEIRO JUNTAR

- 2º Normalizar os dados (1 a 0)
- 3º PCA (usar todos os dados e depois usar o pca para ver quais sao inuties, assim tirar eles)
- 4º fazer as analises como dito no artigo (eu tenho que faze-las
- 5º Organizar o código mais ainda

#add no artigo
- poderia usar o perfil para melhora a recomendação
- recomendaco a aprtir de todos usuarios? ex.: um grupo aprova isso e vc ta no grupo, entao vc vai gostar



